In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
for df in (metric, trace, log, error_logs):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# ---- metric.csv summaries ----
metric_total_rows = int(metric.shape[0])
metric_min_ts = metric['timestamp'].min().isoformat() if metric_total_rows > 0 else None
metric_max_ts = metric['timestamp'].max().isoformat() if metric_total_rows > 0 else None
metric_cmdb_ids = metric['cmdb_id'].dropna().unique()[:20].tolist()

# distinct kpi_name values with counts (top 20)
metric_kpi_counts = metric['kpi_name'].value_counts().reset_index()
metric_kpi_counts.columns = ['kpi_name', 'count']
metric_kpi_counts = metric_kpi_counts.head(20)

# Compute global P95 per (cmdb_id, kpi_name) across full series
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric_p95 = (
    metric.dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(p95=('value', lambda x: float(x.quantile(0.95))))
)
metric_p95_top20 = metric_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

metric_info = {
    'total_rows': metric_total_rows,
    'min_timestamp': metric_min_ts,
    'max_timestamp': metric_max_ts,
    'distinct_cmdb_ids_up_to_20': metric_cmdb_ids
}

# ---- trace.csv summaries ----
trace_total_rows = int(trace.shape[0])
trace_min_ts = trace['timestamp'].min().isoformat() if trace_total_rows > 0 else None
trace_max_ts = trace['timestamp'].max().isoformat() if trace_total_rows > 0 else None
trace_cmdb_ids = trace['cmdb_id'].dropna().unique()[:20].tolist()

trace_trace_counts = trace['trace_name'].value_counts().reset_index()
trace_trace_counts.columns = ['trace_name', 'count']
trace_trace_counts = trace_trace_counts.head(20)

trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace_p95 = (
    trace.dropna(subset=['value'])
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(p95=('value', lambda x: float(x.quantile(0.95))))
)
trace_p95_top20 = trace_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

trace_info = {
    'total_rows': trace_total_rows,
    'min_timestamp': trace_min_ts,
    'max_timestamp': trace_max_ts,
    'distinct_cmdb_ids_up_to_20': trace_cmdb_ids
}

# ---- log.csv summaries ----
log_total_rows = int(log.shape[0])
log_min_ts = log['timestamp'].min().isoformat() if log_total_rows > 0 else None
log_max_ts = log['timestamp'].max().isoformat() if log_total_rows > 0 else None
log_cmdb_ids = log['cmdb_id'].dropna().unique()[:20].tolist()

log_log_counts = log['log_name'].value_counts().reset_index()
log_log_counts.columns = ['log_name', 'count']
log_log_counts = log_log_counts.head(20)

log['value'] = pd.to_numeric(log['value'], errors='coerce')
log_p95 = (
    log.dropna(subset=['value'])
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(p95=('value', lambda x: float(x.quantile(0.95))))
)
log_p95_top20 = log_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

log_info = {
    'total_rows': log_total_rows,
    'min_timestamp': log_min_ts,
    'max_timestamp': log_max_ts,
    'distinct_cmdb_ids_up_to_20': log_cmdb_ids
}

# ---- error_logs.csv summaries ----
error_total_rows = int(error_logs.shape[0])
error_min_ts = error_logs['timestamp'].min().isoformat() if error_total_rows > 0 else None
error_max_ts = error_logs['timestamp'].max().isoformat() if error_total_rows > 0 else None

error_cmdb_counts = error_logs['cmdb_id'].value_counts().reset_index()
error_cmdb_counts.columns = ['cmdb_id', 'count']
error_cmdb_counts = error_cmdb_counts.head(20)

# sample up to 5 rows ordered by timestamp desc; format timestamp as ISO UTC strings
error_samples = (
    error_logs.sort_values('timestamp', ascending=False)
    .head(5)
    .loc[:, ['timestamp', 'cmdb_id', 'message']]
    .copy()
)
error_samples['timestamp'] = error_samples['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

error_info = {
    'total_rows': error_total_rows,
    'min_timestamp': error_min_ts,
    'max_timestamp': error_max_ts
}

# Final compact outputs (variables displayed)
metric_info, metric_kpi_counts.head(20), metric_p95_top20, trace_info, trace_trace_counts.head(20), trace_p95_top20, log_info, log_log_counts.head(20), log_p95_top20, error_info, error_cmdb_counts, error_samples

```
Out[1]:
```
I loaded and summarized the four telemetry files. Key findings:

Metric (metric.csv)
- Total rows: 9,125
- Time range (UTC): 2024-01-24T15:59:00+00:00 → 2024-01-24T16:23:00+00:00
- Distinct cmdb_id (sample, up to 20 shown): ts-admin-basic-info-service, ts-admin-order-service, ts-admin-route-service, ts-admin-travel-service, ts-admin-user-service, … (20 total listed)
- Top kpi_name by count (top entries): cpu (1700), mem (1700), socket (1700), diskio (1675), latency-90 (700), latency-50 (700), workload (700), error (250)
- Global P95 per (cmdb_id, kpi_name) — top entries by P95 (global, descending): 
  - ts-train-service / mem: ~9.95e+08
  - ts-voucher-mysql / mem: ~6.00e+08
  - ts-food-mysql / mem: ~5.99e+08
  - ts-travel2-service / mem: ~2.80e+08
  - ts-travel-service / mem: ~2.72e+08
  (The P95 table is dominated by memory (mem) series with very large values.)

Trace (trace.csv)
- Total rows: 18,680
- Time range (UTC): 2024-01-24T15:59:00+00:00 → 2024-01-24T16:23:00+00:00
- Distinct cmdb_id (sample, up to 20 shown): root, ts-admin-basic-info-service, ts-admin-travel-service, ts-assurance-service, ts-auth-service, …
- Top trace_name by count (top entries): trace.from_root.duration_mean (343), trace.from_root.duration_p95 (343), trace.from_root.error_rate (343), trace.from_root.row_count (343), trace.from_ts-preserve-service.* (several with 258)
- Global P95 per (cmdb_id, trace_name) — top entries by P95 (descending):
  - ts-station-service / trace.from_ts-station-service.row_count: 13040.8
  - ts-station-service / trace.to_ts-station-service.row_count: 13040.8
  - ts-travel2-service / trace.to_ts-travel2-service.row_count: 2794.4
  - ts-train-service / trace.to_ts-train-service.row_count: 2664.8
  - ts-route-service / trace.to_ts-route-service.row_count: 2137.2
  (Top trace P95 values are for row_count-style trace features.)

Log (log.csv)
- Total rows: 2,190
- Time range (UTC): 2024-01-24T15:59:00+00:00 → 2024-01-24T16:23:00+00:00
- Distinct cmdb_id (sample, up to 20 shown): ts-admin-basic-info-service, ts-admin-travel-service, ts-assurance-mongo, ts-assurance-service, ts-auth-mongo, ts-auth-service, …
- Top log_name by count: log.error_count (1095), log.row_count (1095)
- Global P95 per (cmdb_id, log_name) — top entries by P95:
  - ts-basic-service / log.row_count: 2661.2
  - ts-travel2-service / log.row_count: 2432.8
  - ts-seat-service / log.row_count: 1972.4
  - ts-travel-service / log.row_count: 1949.2
  - ts-route-service / log.row_count: 1068.6

Error logs (error_logs.csv)
- Total rows: 0 (file is empty)
- No timestamps, no cmdb_id counts, no sample rows returned.

Notes
- All P95 thresholds were computed globally across the full series for each (cmdb_id, metric_name) as requested.
- Memory (mem) metrics show very large P95 values for several services; trace row_count features also show very large P95s for station/travel-related services.
- error_logs contains no entries in this dataset.

The original code execution output of IPython Kernel is also provided below for reference:

({'total_rows': 9125, 'min_timestamp': '2024-01-24T15:59:00+00:00', 'max_timestamp': '2024-01-24T16:23:00+00:00', 'distinct_cmdb_ids_up_to_20': ['ts-admin-basic-info-service', 'ts-admin-order-service', 'ts-admin-route-service', 'ts-admin-travel-service', 'ts-admin-user-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service', 'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service']},      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4  latency-90    700
5  latency-50    700
6    workload    700
7       error    250,                          cmdb_id kpi_name           p95
0               ts-train-service      mem  9.954589e+08
1               ts-voucher-mysql      mem  6.004163e+08
2                  ts-food-mysql      mem  5.998019e+08
3             ts-travel2-service      mem  2.803427e+08
4              ts-travel-service      mem  2.724164e+08
5         ts-order-other-service      mem  2.718604e+08
6               ts-order-service      mem  2.632534e+08
7             ts-station-service      mem  2.609599e+08
8            ts-food-map-service      mem  2.574481e+08
9               ts-price-service      mem  2.566040e+08
10              ts-route-service      mem  2.551807e+08
11               ts-food-service      mem  2.543718e+08
12  ts-verification-code-service      mem  2.534203e+08
13           ts-contacts-service      mem  2.498864e+08
14              ts-basic-service      mem  2.492377e+08
15             ts-config-service      mem  2.478188e+08
16            ts-consign-service      mem  2.464625e+08
17           ts-security-service      mem  2.460661e+08
18     ts-inside-payment-service      mem  2.444361e+08
19               ts-seat-service      mem  2.443565e+08, {'total_rows': 18680, 'min_timestamp': '2024-01-24T15:59:00+00:00', 'max_timestamp': '2024-01-24T16:23:00+00:00', 'distinct_cmdb_ids_up_to_20': ['root', 'ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-service', 'ts-auth-service', 'ts-basic-service', 'ts-config-service', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-service', 'ts-food-map-service', 'ts-food-service', 'ts-inside-payment-service', 'ts-order-other-service', 'ts-order-service', 'ts-payment-service', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-service', 'ts-route-service']},                                            trace_name  count
0                       trace.from_root.duration_mean    343
1                        trace.from_root.duration_p95    343
2                          trace.from_root.error_rate    343
3                           trace.from_root.row_count    343
4        trace.from_ts-preserve-service.duration_mean    258
5         trace.from_ts-preserve-service.duration_p95    258
6           trace.from_ts-preserve-service.error_rate    258
7            trace.from_ts-preserve-service.row_count    258
8   trace.from_ts-preserve-other-service.duration_...    253
9   trace.from_ts-preserve-other-service.duration_p95    253
10    trace.from_ts-preserve-other-service.error_rate    253
11     trace.from_ts-preserve-other-service.row_count    253
12          trace.to_ts-station-service.duration_mean    172
13           trace.to_ts-station-service.duration_p95    172
14             trace.to_ts-station-service.error_rate    172
15              trace.to_ts-station-service.row_count    172
16      trace.to_ts-order-other-service.duration_mean    170
17       trace.to_ts-order-other-service.duration_p95    170
18         trace.to_ts-order-other-service.error_rate    170
19          trace.to_ts-order-other-service.row_count    170,                   cmdb_id                                  trace_name      p95
0      ts-station-service     trace.from_ts-station-service.row_count  13040.8
1      ts-station-service       trace.to_ts-station-service.row_count  13040.8
2      ts-travel2-service       trace.to_ts-travel2-service.row_count   2794.4
3      ts-travel2-service     trace.from_ts-travel2-service.row_count   2794.4
4        ts-train-service         trace.to_ts-train-service.row_count   2664.8
5        ts-train-service       trace.from_ts-train-service.row_count   2664.8
6        ts-route-service         trace.to_ts-route-service.row_count   2137.2
7        ts-route-service       trace.from_ts-route-service.row_count   2137.2
8        ts-basic-service         trace.to_ts-basic-service.row_count   1928.2
9        ts-basic-service       trace.from_ts-basic-service.row_count   1928.2
10      ts-travel-service      trace.from_ts-travel-service.row_count   1734.6
11      ts-travel-service        trace.to_ts-travel-service.row_count   1734.6
12     ts-station-service       trace.from_ts-basic-service.row_count   1488.6
13       ts-basic-service       trace.to_ts-station-service.row_count   1488.6
14        ts-seat-service        trace.from_ts-seat-service.row_count   1124.0
15        ts-seat-service          trace.to_ts-seat-service.row_count   1124.0
16       ts-basic-service  trace.from_ts-ticketinfo-service.row_count   1050.0
17  ts-ticketinfo-service         trace.to_ts-basic-service.row_count   1050.0
18  ts-ticketinfo-service    trace.to_ts-ticketinfo-service.row_count   1050.0
19  ts-ticketinfo-service  trace.from_ts-ticketinfo-service.row_count   1050.0, {'total_rows': 2190, 'min_timestamp': '2024-01-24T15:59:00+00:00', 'max_timestamp': '2024-01-24T16:23:00+00:00', 'distinct_cmdb_ids_up_to_20': ['ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-basic-service', 'ts-config-mongo', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-notification-mongo']},           log_name  count
0  log.error_count   1095
1    log.row_count   1095,                         cmdb_id       log_name     p95
0              ts-basic-service  log.row_count  2661.2
1            ts-travel2-service  log.row_count  2432.8
2               ts-seat-service  log.row_count  1972.4
3             ts-travel-service  log.row_count  1949.2
4              ts-route-service  log.row_count  1068.6
5        ts-order-other-service  log.row_count   528.2
6              ts-order-service  log.row_count   448.2
7               ts-auth-service  log.row_count   339.6
8               ts-ui-dashboard  log.row_count   296.8
9              ts-price-service  log.row_count   293.2
10          ts-preserve-service  log.row_count   228.7
11    ts-preserve-other-service  log.row_count   168.7
12          ts-security-service  log.row_count    94.8
13              ts-food-service  log.row_count    93.8
14           ts-consign-service  log.row_count    75.6
15          ts-contacts-service  log.row_count    71.1
16      ts-admin-travel-service  log.row_count    32.4
17  ts-admin-basic-info-service  log.row_count    30.8
18          ts-food-map-service  log.row_count    26.6
19    ts-inside-payment-service  log.row_count    21.4, {'total_rows': 0, 'min_timestamp': None, 'max_timestamp': None}, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse existing dataframes: metric, trace, log, metric_p95, trace_p95, log_p95
# If p95 aggregations not present (e.g., variable names differ), compute them (safe reuse).
try:
    metric_p95
except NameError:
    metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
    metric_p95 = (
        metric.dropna(subset=['value'])
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(p95=('value', lambda x: float(x.quantile(0.95))))
    )

try:
    trace_p95
except NameError:
    trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
    trace_p95 = (
        trace.dropna(subset=['value'])
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(p95=('value', lambda x: float(x.quantile(0.95))))
    )

try:
    log_p95
except NameError:
    log['value'] = pd.to_numeric(log['value'], errors='coerce')
    log_p95 = (
        log.dropna(subset=['value'])
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(p95=('value', lambda x: float(x.quantile(0.95))))
    )

# Define incident window (UTC)
window_start = pd.to_datetime("2024-01-24 15:56:04", utc=True)
window_end = pd.to_datetime("2024-01-24 16:26:04", utc=True)

# Filter windows
metric_window = metric[(metric['timestamp'] >= window_start) & (metric['timestamp'] <= window_end)].copy()
trace_window = trace[(trace['timestamp'] >= window_start) & (trace['timestamp'] <= window_end)].copy()
log_window = log[(log['timestamp'] >= window_start) & (log['timestamp'] <= window_end)].copy()

# Counts of rows inside window
metric_window_count = int(metric_window.shape[0])
trace_window_count = int(trace_window.shape[0])
log_window_count = int(log_window.shape[0])

# Helper function to compute per-(cmdb_id, name) stats given window df and p95 df
def compute_window_anomalies(window_df, p95_df, on_cols, value_col='value', ts_col='timestamp', name_col=None):
    # Merge p95 onto window rows
    merged = window_df.merge(p95_df, on=on_cols, how='left')
    # Ensure numeric
    merged[value_col] = pd.to_numeric(merged[value_col], errors='coerce')
    # is_anomaly: value > p95 (if p95 is NaN -> False)
    merged['is_anomaly'] = (merged[value_col] > merged['p95']).fillna(False)
    # anomaly timestamp or NaT
    merged['anomaly_ts'] = merged[ts_col].where(merged['is_anomaly'], pd.NaT)

    # Aggregate stats from window
    agg = (
        merged.groupby(on_cols, as_index=False)
        .agg(
            total_points_in_window=(value_col, 'count'),
            anomalies_count=('is_anomaly', 'sum'),
            earliest_anomaly_ts=('anomaly_ts', 'min'),
            max_value_in_window=(value_col, 'max'),
        )
    )

    # Now join with full p95_df to include groups with zero points in window
    full = p95_df.merge(agg, on=on_cols, how='left')
    # Fill zeros for groups with no points
    full['total_points_in_window'] = full['total_points_in_window'].fillna(0).astype(int)
    full['anomalies_count'] = full['anomalies_count'].fillna(0).astype(int)
    # earliest_anomaly_ts keep NaT where no anomaly
    # max_value_in_window stays NaN where no points

    # severity_ratio: max_value_in_window / p95, if p95==0 -> None, if max_value_in_window NaN -> None
    def compute_sev(row):
        p = row['p95']
        mv = row['max_value_in_window']
        if pd.isna(mv) or pd.isna(p):
            return None
        if p == 0:
            return None
        return float(mv / p)

    full['severity_ratio'] = full.apply(compute_sev, axis=1)

    # Format earliest_anomaly_ts as ISO Z string or None
    def fmt_ts(x):
        if pd.isna(x):
            return None
        # ensure UTC and format Z
        return pd.to_datetime(x).strftime('%Y-%m-%dT%H:%M:%SZ')

    full['earliest_anomaly_timestamp'] = full['earliest_anomaly_ts'].apply(fmt_ts)
    # Drop internal earliest_anomaly_ts column
    full = full.drop(columns=['earliest_anomaly_ts'])

    # Filter only groups with total_points_in_window > 0 for return (but full contains all)
    with_points = full[full['total_points_in_window'] > 0].copy()

    # Sorting by anomalies_count desc then severity_ratio desc (None -> treated as -inf for sorting)
    with_points['severity_for_sort'] = with_points['severity_ratio'].apply(lambda x: x if x is not None else -np.inf)
    with_points = with_points.sort_values(['anomalies_count', 'severity_for_sort'], ascending=[False, False])
    with_points = with_points.drop(columns=['severity_for_sort'])

    # Select and order columns for readability: on_cols..., total_points_in_window, anomalies_count, earliest_anomaly_timestamp, max_value_in_window, p95, severity_ratio
    display_cols = on_cols + [
        'total_points_in_window',
        'anomalies_count',
        'earliest_anomaly_timestamp',
        'max_value_in_window',
        'p95',
        'severity_ratio',
    ]
    result_top = with_points.loc[:, display_cols].head(20).reset_index(drop=True)

    # Compute summary counts and distinct cmdb_ids with anomalies
    anomalies_groups_count = int((full['anomalies_count'] > 0).sum())
    anomaly_cmdbs = full.loc[full['anomalies_count'] > 0, on_cols[0]].dropna().unique()[:20].tolist()

    return full, result_top, anomalies_groups_count, anomaly_cmdbs

# Metric anomalies
metric_full_stats, metric_anomalies_top20, metric_anomaly_groups_count, metric_anomaly_cmdbs = compute_window_anomalies(
    metric_window,
    metric_p95,
    on_cols=['cmdb_id', 'kpi_name'],
    value_col='value',
    ts_col='timestamp'
)

# Trace anomalies
trace_full_stats, trace_anomalies_top20, trace_anomaly_groups_count, trace_anomaly_cmdbs = compute_window_anomalies(
    trace_window,
    trace_p95,
    on_cols=['cmdb_id', 'trace_name'],
    value_col='value',
    ts_col='timestamp'
)

# Log anomalies
log_full_stats, log_anomalies_top20, log_anomaly_groups_count, log_anomaly_cmdbs = compute_window_anomalies(
    log_window,
    log_p95,
    on_cols=['cmdb_id', 'log_name'],
    value_col='value',
    ts_col='timestamp'
)

# Final compact outputs (JSON-serializable structures via DataFrames and lists)
metric_window_count, metric_anomalies_top20, metric_anomaly_groups_count, metric_anomaly_cmdbs, \
trace_window_count, trace_anomalies_top20, trace_anomaly_groups_count, trace_anomaly_cmdbs, \
log_window_count, log_anomalies_top20, log_anomaly_groups_count, log_anomaly_cmdbs

```
Out[2]:
```
Summary of incident-window analysis (window = 2024-01-24 15:56:04 → 2024-01-24 16:26:04 UTC)

Overall rows in window
- metric.csv: 9,125 rows
- trace.csv: 18,680 rows
- log.csv: 2,190 rows

Metric anomalies (per (cmdb_id, kpi_name))
- Number of (cmdb_id, kpi_name) series that had at least one anomaly in the window: 318
- Distinct cmdb_ids with metric anomalies (sample up to 20): 
  ts-admin-basic-info-service, ts-admin-order-service, ts-admin-route-service, ts-admin-travel-service, ts-admin-user-service, ts-assurance-mongo, ts-assurance-service, ts-auth-mongo, ts-auth-service, ts-avatar-service, ts-basic-service, ts-cancel-service, ts-config-mongo, ts-config-service, ts-consign-mongo, ts-consign-price-mongo, ts-consign-price-service, ts-consign-service, ts-contacts-mongo, ts-contacts-service
- Top metric anomaly rows (top 5 by anomalies_count then severity_ratio):
  1. ts-admin-user-service / cpu — total_points_in_window 25, anomalies_count 2, earliest anomaly 2024-01-24T16:13:00Z, max_value 0.306679, global_p95 0.110982, severity_ratio ≈ 2.76
  2. ts-admin-route-service / cpu — 25 pts, 2 anomalies, earliest 2024-01-24T16:05:00Z, severity_ratio ≈ 2.64
  3. ts-route-plan-service / cpu — 25 pts, 2 anomalies, earliest 2024-01-24T16:13:00Z, severity_ratio ≈ 2.50
  4. ts-travel-plan-service / cpu — 25 pts, 2 anomalies, earliest 2024-01-24T16:14:00Z, severity_ratio ≈ 2.47
  5. ts-rebook-service / cpu — 25 pts, 2 anomalies, earliest 2024-01-24T16:14:00Z, severity_ratio ≈ 2.23
- Observations: CPU KPIs dominate the top metric anomalies; earliest top anomalies occur around 16:05–16:14 UTC.

Trace anomalies (per (cmdb_id, trace_name))
- Number of (cmdb_id, trace_name) series with at least one anomaly in the window: 570
- Distinct cmdb_ids with trace anomalies (sample up to 20): 
  root, ts-admin-basic-info-service, ts-admin-travel-service, ts-assurance-service, ts-auth-service, ts-basic-service, ts-config-service, ts-consign-price-service, ts-consign-service, ts-contacts-service, ts-food-map-service, ts-food-service, ts-inside-payment-service, ts-order-other-service, ts-order-service, ts-payment-service, ts-preserve-other-service, ts-preserve-service, ts-price-service, ts-route-service
- Top trace anomaly rows (top 5 by anomalies_count then severity_ratio):
  1. ts-consign-price-service / trace.from_ts-consign-price-service.duration_p95 — 25 pts, 2 anomalies, earliest 2024-01-24T16:15:00Z, max_value 0.165184, global_p95 0.006428, severity_ratio ≈ 25.70
  2. ts-consign-price-service / trace.to_ts-consign-price-service.duration_p95 — same values and severity ≈ 25.70
  3. ts-auth-service / trace.from_ts-auth-service.duration_p95 — 25 pts, 2 anomalies, earliest 2024-01-24T16:15:00Z, severity_ratio ≈ 16.47
  4. ts-auth-service / trace.to_ts-auth-service.duration_p95 — same as above
  5. ts-preserve-service / trace.to_ts-seat-service.duration_p95 — 24 pts, 2 anomalies, earliest 2024-01-24T16:17:00Z, severity_ratio ≈ 10.72
- Observations: Trace-duration P95 series show very large severity ratios (some >> 10), especially for ts-consign-price-service and ts-auth-service, indicating large deviations from their global P95.

Log anomalies (per (cmdb_id, log_name))
- Number of (cmdb_id, log_name) series with at least one anomaly in the window: 21
- Distinct cmdb_ids with log anomalies (up to 20): 
  ts-admin-basic-info-service, ts-admin-travel-service, ts-assurance-service, ts-auth-service, ts-basic-service, ts-consign-service, ts-contacts-service, ts-food-map-service, ts-food-service, ts-inside-payment-service, ts-order-other-service, ts-order-service, ts-preserve-other-service, ts-preserve-service, ts-price-service, ts-route-service, ts-seat-service, ts-security-service, ts-travel-service, ts-travel2-service
- Top log anomaly rows (top 5):
  1. ts-preserve-other-service / log.row_count — 24 pts, 2 anomalies, earliest 2024-01-24T16:02:00Z, max_value 253, global_p95 168.7, severity_ratio ≈ 1.50
  2. ts-consign-service / log.row_count — 25 pts, 2 anomalies, earliest 2024-01-24T16:14:00Z, severity_ratio ≈ 1.20
  3. ts-price-service / log.row_count — 25 pts, 2 anomalies, earliest 2024-01-24T16:03:00Z, severity_ratio ≈ 1.17
  4. ts-seat-service / log.row_count — 25 pts, 2 anomalies, earliest 2024-01-24T16:03:00Z, severity_ratio ≈ 1.17
  5. ts-basic-service / log.row_count — 25 pts, 2 anomalies, earliest 2024-01-24T16:03:00Z, severity_ratio ≈ 1.17
- Observations: Log anomalies mostly appear as elevated log.row_count values with modest severity ratios (~1.03–1.50).

Bottom-line takeaway
- There are widespread anomalies across metrics and traces in the window: 318 metric-series anomalies and 570 trace-series anomalies; log anomalies are fewer (21 series).
- Trace anomalies show the largest relative deviations from global P95 (very high severity_ratios for ts-consign-price-service and ts-auth-service).
- Metric anomalies are concentrated in CPU KPIs across many services (top CPU anomalies with severity ratios ~2–2.8).
- Log anomalies are present but with smaller relative increases (severity_ratios ~1.03–1.5).
- Earliest top anomaly times: some log anomalies appear around ~16:02–16:03 UTC, metric CPU anomalies around ~16:05–16:14 UTC, and many trace-duration anomalies at ~16:15–16:17 UTC.

If you want, I can:
- produce a ranked list of the top N affected cmdb_ids across all telemetry types combined, or
- expand the returned lists beyond the top 20 for deeper investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(9125,                          cmdb_id    kpi_name  total_points_in_window  anomalies_count earliest_anomaly_timestamp  max_value_in_window       p95  severity_ratio
0          ts-admin-user-service         cpu                      25                2       2024-01-24T16:13:00Z             0.306679  0.110982        2.763325
1         ts-admin-route-service         cpu                      25                2       2024-01-24T16:05:00Z             0.309976  0.117205        2.644732
2          ts-route-plan-service         cpu                      25                2       2024-01-24T16:13:00Z             0.267191  0.106757        2.502798
3         ts-travel-plan-service         cpu                      25                2       2024-01-24T16:14:00Z             0.288717  0.116986        2.467963
4              ts-rebook-service         cpu                      25                2       2024-01-24T16:14:00Z             0.281189  0.126371        2.225117
5         ts-admin-order-service         cpu                      25                2       2024-01-24T16:13:00Z             0.254273  0.115425        2.202927
6              ts-cancel-service         cpu                      25                2       2024-01-24T16:13:00Z             0.264964  0.126550        2.093756
7        ts-notification-service         cpu                      25                2       2024-01-24T16:14:00Z             0.282831  0.138458        2.042722
8             ts-execute-service         cpu                      25                2       2024-01-24T16:13:00Z             0.271145  0.139304        1.946434
9                ts-user-service  latency-90                      25                2       2024-01-24T16:16:00Z             0.049713  0.025818        1.925522
10              ts-price-service         cpu                      25                2       2024-01-24T16:12:00Z             8.093842  4.480958        1.806275
11           ts-food-map-service         cpu                      25                2       2024-01-24T16:13:00Z             2.946607  1.685803        1.747895
12  ts-verification-code-service         cpu                      25                2       2024-01-24T15:59:00Z             0.266945  0.154734        1.725193
13              ts-order-service  latency-90                      25                2       2024-01-24T16:02:00Z             0.080560  0.049323        1.633318
14     ts-preserve-other-service         cpu                      25                2       2024-01-24T16:06:00Z             2.757640  1.758167        1.568474
15      ts-consign-price-service    workload                      25                2       2024-01-24T16:15:00Z             0.533000  0.370080        1.440229
16        ts-order-other-service  latency-90                      25                2       2024-01-24T16:11:00Z             0.026071  0.018129        1.438096
17               ts-auth-service  latency-90                      25                2       2024-01-24T16:15:00Z             1.235899  0.862761        1.432492
18           ts-preserve-service    workload                      25                2       2024-01-24T16:09:00Z             0.332350  0.232837        1.427395
19            ts-consign-service         cpu                      25                2       2024-01-24T16:15:00Z             1.466444  1.027621        1.427028, 318, ['ts-admin-basic-info-service', 'ts-admin-order-service', 'ts-admin-route-service', 'ts-admin-travel-service', 'ts-admin-user-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service', 'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service'], 18680,                       cmdb_id                                         trace_name  total_points_in_window  anomalies_count earliest_anomaly_timestamp  max_value_in_window       p95  severity_ratio
0    ts-consign-price-service   trace.from_ts-consign-price-service.duration_p95                      25                2       2024-01-24T16:15:00Z             0.165184  0.006428       25.697276
1    ts-consign-price-service     trace.to_ts-consign-price-service.duration_p95                      25                2       2024-01-24T16:15:00Z             0.165184  0.006428       25.697276
2             ts-auth-service            trace.from_ts-auth-service.duration_p95                      25                2       2024-01-24T16:15:00Z             0.102700  0.006235       16.472629
3             ts-auth-service              trace.to_ts-auth-service.duration_p95                      25                2       2024-01-24T16:15:00Z             0.102700  0.006235       16.472629
4         ts-preserve-service              trace.to_ts-seat-service.duration_p95                      24                2       2024-01-24T16:17:00Z             1.062394  0.099104       10.720028
5             ts-seat-service        trace.from_ts-preserve-service.duration_p95                      24                2       2024-01-24T16:17:00Z             1.062394  0.099104       10.720028
6   ts-inside-payment-service      trace.to_ts-order-other-service.duration_mean                      22                2       2024-01-24T16:10:00Z             0.065577  0.006245       10.501001
7      ts-order-other-service  trace.from_ts-inside-payment-service.duration_...                      22                2       2024-01-24T16:10:00Z             0.065577  0.006245       10.501001
8            ts-order-service          trace.from_ts-travel-service.duration_p95                      25                2       2024-01-24T16:05:00Z             0.135579  0.017739        7.643036
9           ts-travel-service             trace.to_ts-order-service.duration_p95                      25                2       2024-01-24T16:05:00Z             0.135579  0.017739        7.643036
10   ts-consign-price-service  trace.from_ts-consign-price-service.duration_mean                      25                2       2024-01-24T16:15:00Z             0.020235  0.003033        6.672309
11   ts-consign-price-service    trace.to_ts-consign-price-service.duration_mean                      25                2       2024-01-24T16:15:00Z             0.020235  0.003033        6.672309
12  ts-preserve-other-service        trace.to_ts-ticketinfo-service.duration_p95                      24                2       2024-01-24T16:11:00Z             0.723857  0.109016        6.639893
13      ts-ticketinfo-service  trace.from_ts-preserve-other-service.duration_p95                      24                2       2024-01-24T16:11:00Z             0.723857  0.109016        6.639893
14           ts-order-service           trace.from_ts-order-service.duration_p95                      25                2       2024-01-24T16:17:00Z             0.080363  0.012582        6.387286
15           ts-order-service             trace.to_ts-order-service.duration_p95                      25                2       2024-01-24T16:17:00Z             0.080363  0.012582        6.387286
16  ts-inside-payment-service       trace.to_ts-order-other-service.duration_p95                      22                2       2024-01-24T16:10:00Z             0.068678  0.010876        6.314763
17     ts-order-other-service  trace.from_ts-inside-payment-service.duration_p95                      22                2       2024-01-24T16:10:00Z             0.068678  0.010876        6.314763
18   ts-consign-price-service         trace.from_ts-consign-service.duration_p95                      25                2       2024-01-24T16:12:00Z             0.106162  0.017895        5.932600
19         ts-consign-service     trace.to_ts-consign-price-service.duration_p95                      25                2       2024-01-24T16:12:00Z             0.106162  0.017895        5.932600, 570, ['root', 'ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-service', 'ts-auth-service', 'ts-basic-service', 'ts-config-service', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-service', 'ts-food-map-service', 'ts-food-service', 'ts-inside-payment-service', 'ts-order-other-service', 'ts-order-service', 'ts-payment-service', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-service', 'ts-route-service'], 2190,                         cmdb_id       log_name  total_points_in_window  anomalies_count earliest_anomaly_timestamp  max_value_in_window     p95  severity_ratio
0     ts-preserve-other-service  log.row_count                      24                2       2024-01-24T16:02:00Z                  253   168.7        1.499704
1            ts-consign-service  log.row_count                      25                2       2024-01-24T16:14:00Z                   91    75.6        1.203704
2              ts-price-service  log.row_count                      25                2       2024-01-24T16:03:00Z                  344   293.2        1.173261
3               ts-seat-service  log.row_count                      25                2       2024-01-24T16:03:00Z                 2304  1972.4        1.168120
4              ts-basic-service  log.row_count                      25                2       2024-01-24T16:03:00Z                 3107  2661.2        1.167518
5           ts-food-map-service  log.row_count                      25                2       2024-01-24T16:08:00Z                   31    26.6        1.165414
6           ts-preserve-service  log.row_count                      24                2       2024-01-24T16:08:00Z                  265   228.7        1.158723
7               ts-food-service  log.row_count                      25                2       2024-01-24T16:08:00Z                  107    93.8        1.140725
8              ts-route-service  log.row_count                      25                2       2024-01-24T16:03:00Z                 1207  1068.6        1.129515
9        ts-order-other-service  log.row_count                      25                2       2024-01-24T16:06:00Z                  592   528.2        1.120788
10           ts-travel2-service  log.row_count                      25                2       2024-01-24T16:03:00Z                 2705  2432.8        1.111888
11          ts-contacts-service  log.row_count                      24                2       2024-01-24T16:06:00Z                   78    71.1        1.097046
12          ts-security-service  log.row_count                      24                2       2024-01-24T16:06:00Z                  104    94.8        1.097046
13            ts-travel-service  log.row_count                      25                2       2024-01-24T16:06:00Z                 2101  1949.2        1.077878
14              ts-auth-service  log.row_count                      25                2       2024-01-24T16:00:00Z                  358   339.6        1.054181
15      ts-admin-travel-service  log.row_count                      25                2       2024-01-24T16:05:00Z                   34    32.4        1.049383
16             ts-order-service  log.row_count                      25                2       2024-01-24T16:06:00Z                  470   448.2        1.048639
17  ts-admin-basic-info-service  log.row_count                      25                2       2024-01-24T16:02:00Z                   32    30.8        1.038961
18              ts-ui-dashboard  log.row_count                      25                2       2024-01-24T16:06:00Z                  307   296.8        1.034367
19    ts-inside-payment-service  log.row_count                      24                2       2024-01-24T16:17:00Z                   22    21.4        1.028037, 21, ['ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-service', 'ts-auth-service', 'ts-basic-service', 'ts-consign-service', 'ts-contacts-service', 'ts-food-map-service', 'ts-food-service', 'ts-inside-payment-service', 'ts-order-other-service', 'ts-order-service', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-service', 'ts-route-service', 'ts-seat-service', 'ts-security-service', 'ts-travel-service', 'ts-travel2-service'])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing dataframes and p95s: metric, trace, log, metric_p95, trace_p95, log_p95, metric_window, trace_window, log_window
candidates = [
    "ts-consign-price-service",
    "ts-auth-service",
    "ts-consign-service",
    "ts-preserve-service",
    "ts-order-service",
    "ts-admin-user-service",
    "ts-admin-route-service",
    "ts-price-service",
    "ts-route-plan-service",
    "ts-travel-plan-service",
]

window_start, window_end  # reuse existing

def longest_consecutive_run(tstamps):
    # tstamps: list/array of Timestamps (pd.Timestamp), assume sorted ascending unique
    if len(tstamps) == 0:
        return 0
    # convert to integer seconds
    secs = np.array([int(ts.value // 10**9) for ts in tstamps], dtype=np.int64)
    if len(secs) == 1:
        return 1
    # diffs in seconds
    diffs = np.diff(secs)
    # consecutive where diff == 60 (one minute)
    runs = []
    run_len = 1
    for d in diffs:
        if d == 60:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return int(max(runs)) if runs else int(run_len)

def format_ts_list(ts_series, limit=10):
    # ts_series: pd.Series of timestamps
    ts_series = ts_series.dropna().sort_values().unique()
    out = []
    for ts in ts_series[:limit]:
        out.append(pd.to_datetime(ts).strftime("%Y-%m-%dT%H:%M:%SZ"))
    return out

results = []

for cid in candidates:
    # Metric faults for candidate
    p95_metric_c = metric_p95[metric_p95['cmdb_id'] == cid].copy()
    window_metric_c = metric_window[metric_window['cmdb_id'] == cid].copy()
    if not p95_metric_c.empty and not window_metric_c.empty:
        merged = window_metric_c.merge(p95_metric_c, on=['cmdb_id', 'kpi_name'], how='right', suffixes=('', '_p95'))
        # merged may include p95 rows with no window points (NaN value) because of right join; keep only rows present in window by dropping rows where timestamp isna
        # But we need to identify kpi_names that have at least one anomaly in window: compute per kpi_name using inner merge of actual window rows with p95
        merged_actual = window_metric_c.merge(p95_metric_c, on=['cmdb_id', 'kpi_name'], how='inner')
        merged_actual['value'] = pd.to_numeric(merged_actual['value'], errors='coerce')
        merged_actual['is_anomaly'] = (merged_actual['value'] > merged_actual['p95']).fillna(False)
        metric_groups = []
        for kpi, g in merged_actual.groupby('kpi_name'):
            total_points = int(g.shape[0])
            anomalies = g[g['is_anomaly']]
            anomalies_count = int(anomalies.shape[0])
            if anomalies_count == 0:
                continue
            anomaly_ts_list = format_ts_list(anomalies['timestamp'], limit=10)
            earliest_anom = anomaly_ts_list[0] if len(anomaly_ts_list)>0 else None
            max_value = float(g['value'].max()) if not g['value'].isna().all() else None
            global_p95 = float(p95_metric_c.loc[p95_metric_c['kpi_name']==kpi, 'p95'].iloc[0]) if kpi in p95_metric_c['kpi_name'].values else None
            severity = None
            if global_p95 is not None and not np.isnan(global_p95) and global_p95 != 0 and max_value is not None:
                severity = float(max_value / global_p95)
            # compute longest consecutive anomaly run
            longest_run = longest_consecutive_run(pd.to_datetime(anomalies['timestamp']).sort_values().unique())
            metric_groups.append({
                'kpi_name': kpi,
                'total_points_in_window': total_points,
                'anomalies_count': anomalies_count,
                'anomaly_timestamps': anomaly_ts_list,
                'longest_consecutive_anomaly_run_length': longest_run,
                'earliest_anomaly_timestamp': earliest_anom,
                'max_value_in_window': max_value,
                'global_p95': global_p95,
                'severity_ratio': severity
            })
        # sort by severity desc (None -> -inf) and take top 10
        metric_groups_sorted = sorted(metric_groups, key=lambda x: (x['severity_ratio'] if x['severity_ratio'] is not None else -np.inf), reverse=True)[:10]
    else:
        metric_groups_sorted = []

    # Trace faults for candidate
    p95_trace_c = trace_p95[trace_p95['cmdb_id'] == cid].copy()
    window_trace_c = trace_window[trace_window['cmdb_id'] == cid].copy()
    if not p95_trace_c.empty and not window_trace_c.empty:
        merged_actual = window_trace_c.merge(p95_trace_c, on=['cmdb_id', 'trace_name'], how='inner')
        merged_actual['value'] = pd.to_numeric(merged_actual['value'], errors='coerce')
        merged_actual['is_anomaly'] = (merged_actual['value'] > merged_actual['p95']).fillna(False)
        trace_groups = []
        for tname, g in merged_actual.groupby('trace_name'):
            anomalies = g[g['is_anomaly']]
            anomalies_count = int(anomalies.shape[0])
            if anomalies_count == 0:
                continue
            total_points = int(g.shape[0])
            anomaly_ts_list = format_ts_list(anomalies['timestamp'], limit=10)
            earliest_anom = anomaly_ts_list[0] if len(anomaly_ts_list)>0 else None
            max_value = float(g['value'].max()) if not g['value'].isna().all() else None
            global_p95 = float(p95_trace_c.loc[p95_trace_c['trace_name']==tname, 'p95'].iloc[0]) if tname in p95_trace_c['trace_name'].values else None
            severity = None
            if global_p95 is not None and not np.isnan(global_p95) and global_p95 != 0 and max_value is not None:
                severity = float(max_value / global_p95)
            longest_run = longest_consecutive_run(pd.to_datetime(anomalies['timestamp']).sort_values().unique())
            trace_groups.append({
                'trace_name': tname,
                'total_points_in_window': total_points,
                'anomalies_count': anomalies_count,
                'anomaly_timestamps': anomaly_ts_list,
                'longest_consecutive_anomaly_run_length': longest_run,
                'earliest_anomaly_timestamp': earliest_anom,
                'max_value_in_window': max_value,
                'global_p95': global_p95,
                'severity_ratio': severity
            })
        trace_groups_sorted = sorted(trace_groups, key=lambda x: (x['severity_ratio'] if x['severity_ratio'] is not None else -np.inf), reverse=True)[:10]
    else:
        trace_groups_sorted = []

    # Linking traces: any trace_window rows (across any cmdb_id) where trace_name contains from_ts-<cid> or to_ts-<cid>
    pattern_from = f"from_ts-{cid}"
    pattern_to = f"to_ts-{cid}"
    linking_rows = trace_window[
        trace_window['trace_name'].str.contains(pattern_from) | trace_window['trace_name'].str.contains(pattern_to)
    ].copy()
    if not linking_rows.empty:
        # Merge global p95 for each (cmdb_id, trace_name)
        linking_merged = linking_rows.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')
        linking_merged['value'] = pd.to_numeric(linking_merged['value'], errors='coerce')
        linking_merged['is_anomaly'] = (linking_merged['value'] > linking_merged['p95']).fillna(False)
        # select and format columns, sort chronological asc, limit 20
        linking_merged_sorted = linking_merged.sort_values('timestamp', ascending=True).head(20).copy()
        linking_merged_sorted['timestamp'] = linking_merged_sorted['timestamp'].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        linking_traces_list = linking_merged_sorted.loc[:, ['timestamp', 'cmdb_id', 'trace_name', 'value', 'p95', 'is_anomaly']].to_dict(orient='records')
    else:
        linking_traces_list = []

    # Log faults for candidate
    p95_log_c = log_p95[log_p95['cmdb_id'] == cid].copy()
    window_log_c = log_window[log_window['cmdb_id'] == cid].copy()
    if not p95_log_c.empty and not window_log_c.empty:
        merged_actual = window_log_c.merge(p95_log_c, on=['cmdb_id', 'log_name'], how='inner')
        merged_actual['value'] = pd.to_numeric(merged_actual['value'], errors='coerce')
        merged_actual['is_anomaly'] = (merged_actual['value'] > merged_actual['p95']).fillna(False)
        log_groups = []
        for lname, g in merged_actual.groupby('log_name'):
            anomalies = g[g['is_anomaly']]
            anomalies_count = int(anomalies.shape[0])
            if anomalies_count == 0:
                continue
            total_points = int(g.shape[0])
            anomaly_ts_list = format_ts_list(anomalies['timestamp'], limit=10)
            earliest_anom = anomaly_ts_list[0] if len(anomaly_ts_list)>0 else None
            max_value = float(g['value'].max()) if not g['value'].isna().all() else None
            global_p95 = float(p95_log_c.loc[p95_log_c['log_name']==lname, 'p95'].iloc[0]) if lname in p95_log_c['log_name'].values else None
            severity = None
            if global_p95 is not None and not np.isnan(global_p95) and global_p95 != 0 and max_value is not None:
                severity = float(max_value / global_p95)
            log_groups.append({
                'log_name': lname,
                'total_points_in_window': total_points,
                'anomalies_count': anomalies_count,
                'anomaly_timestamps': anomaly_ts_list,
                'earliest_anomaly_timestamp': earliest_anom,
                'max_value_in_window': max_value,
                'global_p95': global_p95,
                'severity_ratio': severity
            })
        log_groups_sorted = sorted(log_groups, key=lambda x: (x['severity_ratio'] if x['severity_ratio'] is not None else -np.inf), reverse=True)[:5]
    else:
        log_groups_sorted = []

    results.append({
        'cmdb_id': cid,
        'metric_faults': metric_groups_sorted,
        'trace_faults': trace_groups_sorted,
        'linking_traces': linking_traces_list,
        'log_faults': log_groups_sorted
    })

# Display the compact results variable
results

```
Out[3]:
```
Summary focused on the 10 candidate services (window 2024-01-24 15:56:04 → 2024-01-24 16:26:04 UTC)

Top-level finding (prioritized by observed severity):
- The strongest signals point to ts-consign-price-service and ts-auth-service as the most severe and likely involved components:
  - ts-consign-price-service shows very large trace-duration P95 spikes (severity ≈ 25.7) at ~16:15 and ~16:22 UTC, plus multiple metric anomalies (workload, latency, CPU, diskio, mem).
  - ts-auth-service shows very large trace-duration P95 spikes (severity ≈ 16.5) at ~16:15–16:23 UTC, plus elevated latency and CPU metrics.

Detailed per-candidate concise summaries (most important anomalies only)

1) ts-consign-price-service
- Metrics: multiple KPI anomalies (workload severity ≈ 1.44, latency-90 ≈ 1.25, cpu ≈ 1.15, diskio ≈ 1.05, mem ≈ 1.00025). Earliest relevant anomalies around 16:03 and stronger around 16:15–16:23 UTC. No long consecutive runs (mostly isolated 1-minute anomalies, except latency-90 had a 2-minute run).
- Traces: very large duration P95 anomalies:
  - trace.from_ts-consign-price-service.duration_p95 and trace.to_ts-consign-price-service.duration_p95 — severity ≈ 25.7, earliest anomaly 2024-01-24T16:15:00Z.
  - duration_mean variants also elevated (severity ≈ 6.67).
  - Also trace entries referencing consign-service show elevated durations (severity ≈ 5.93).
- Linking traces: none returned by the cross-service linking query (no additional external traces matched the pattern), but trace_faults are present on the service itself, indicating large internal call latency.
- Logs: no anomalous log rows for this candidate in window.

Interpretation: very strong trace-duration degradation inside ts-consign-price-service at ~16:15 and ~16:22 UTC — high priority for investigation.

2) ts-auth-service
- Metrics: latency-90 (severity ≈ 1.43), latency-50, cpu all elevated; mem and diskio slightly above P95. Earliest anomalies around 16:05 and main spikes at 16:15–16:23 UTC.
- Traces: very large trace-duration P95 spikes:
  - trace.from_ts-auth-service.duration_p95 and trace.to_ts-auth-service.duration_p95 — severity ≈ 16.47, earliest 16:15 UTC.
  - duration_mean variants severity ≈ 3.0.
  - trace.from_root.duration_mean / duration_p95 also show anomalies (some root-level latency increases).
- Linking traces: none returned by the cross-service linking query.
- Logs: log.row_count anomaly (severity ≈ 1.054) at ~16:00–16:03 UTC (two points).

Interpretation: strong trace-duration anomalies centered on auth calls (16:15–16:23). Auth is a high-priority suspect for latency-related root cause or upstream timing pressure.

3) ts-consign-service
- Metrics: CPU (severity ≈ 1.43), workload ≈ 1.35, others mild. Earliest anomalies ~16:02–16:15 UTC.
- Traces: calls to/from consign-price-service are anomalous (e.g., to_ts-consign-price-service.duration_p95 severity ≈ 5.93; from/to duration_mean also elevated).
- Logs: log.row_count elevated (severity ≈ 1.20) at ~16:14–16:17 UTC.

Interpretation: consign-service shows metric and trace anomalies and appears linked to consign-price-service (mutual trace spikes). It may be involved in the same fault domain as consign-price-service.

4) ts-preserve-service
- Metrics: workload (severity ≈ 1.43) and CPU (severity ≈ 1.18) with earliest anomalies ~16:09. CPU shows a 2-minute consecutive run for one KPI.
- Traces: very large trace-duration anomalies to seat-service (trace.to_ts-seat-service.duration_p95 severity ≈ 10.72 at ~16:17 and ~16:22 UTC) and other to_* traces elevated. Several row_count anomalies (some 2-minute runs).
- Logs: log.row_count anomaly (severity ≈ 1.16) at ~16:06–16:09 UTC.

Interpretation: preserve-service shows strong upward trace durations toward seat-service — a high-severity trace-related signal (possible downstream slowdown or heavy load affecting flows).

5) ts-order-service
- Metrics: latency-90 highest (severity ≈ 1.63), CPU, and a 2-minute workload run. Mem and diskio slightly high later (~16:22–16:23).
- Traces: very large trace-duration anomalies from travel-service (severity ≈ 7.64) and from/to order-service duration_p95 (severity ≈ 6.39). Some from-inside-payment traces elevated.
- Logs: log.row_count normal-ish (no critical logs for order-service in this candidate set beyond small anomalies).

Interpretation: order-service has strong trace-duration anomalies indicating request latencies in flows involving travel-service and inside-payment paths.

6) ts-admin-user-service
- Metrics: CPU is most notable (severity ≈ 2.76) with a 2-minute consecutive anomaly at ~16:13–16:14 UTC; diskio/socket/mem mildly above p95.
- Traces: none flagged in the candidate trace faults (no trace anomalies reported).
- Logs: none.

Interpretation: admin-user shows a strong CPU spike ~16:13–16:14 but no trace linkage in this dataset. This could be a local CPU issue or workload burst.

7) ts-admin-route-service
- Metrics: CPU severity ≈ 2.64 (earliest 16:05–16:14), other small metric anomalies.
- Traces/logs: none flagged for trace or logs.

Interpretation: admin-route has a notable CPU spike ~16:05 but no trace evidence to link it to other failures.

8) ts-price-service
- Metrics: CPU severity ≈ 1.81 (2-minute run at ~16:12–16:13), other KPIs mildly elevated.
- Traces: calls from admin-basic-info and basic-service to price-service show elevated durations (trace.from_ts-admin-basic-info.duration_p95 severity ≈ 5.28). Many trace row_count anomalies around ~16:03–16:06.
- Logs: log.row_count elevated (severity ≈ 1.17) at ~16:03–16:06 UTC.

Interpretation: price-service shows both CPU and trace-related issues (incoming call durations elevated), worth investigating as part of the latency propagation chain.

9) ts-route-plan-service
- Metrics: CPU severity ≈ 2.50 (2-minute run ~16:13–16:14); minor diskio/socket/mem anomalies.
- Traces/logs: none flagged.

Interpretation: local CPU spike similar to other admin/plan services; no trace linking evidence.

10) ts-travel-plan-service
- Metrics: CPU severity ≈ 2.47 (2-minute run ~16:14–16:15); minor diskio/socket/mem anomalies.
- Traces/logs: none flagged.

Interpretation: local CPU spike; no trace linking evidence.

Cross-candidate linking and timeline highlights
- Strongest trace-duration anomalies cluster around ~16:15 UTC (ts-consign-price-service and ts-auth-service) and ~16:17–16:22 UTC (preserve-service -> seat-service).
- Several services (consign-price, consign-service, preserve-service, order-service, auth-service, price-service) show both trace-duration and metric anomalies — these are the highest-priority cluster for cross-service fault propagation.
- Admin/user/route/route-plan/travel-plan show prominent CPU spikes (severity ~2.4–2.8) but little trace evidence tying them into the same cross-service trace-duration spikes — these look like local CPU/workload bursts.

Suggested next steps (practical triage)
1. Immediately investigate ts-consign-price-service and ts-auth-service:
   - Collect full traces (span-level details) around 16:12–16:23 UTC for calls in/out of these services to see which spans are slow and where latency originates.
   - Check service logs, thread pools, DB calls, and external dependencies at those timestamps.
2. Inspect ts-consign-service and ts-preserve-service next (they show linked trace anomalies with consign-price and seat-service respectively).
3. For services with large CPU spikes (ts-admin-user-service, ts-admin-route-service, ts-route-plan-service, ts-travel-plan-service): verify if they are suffering high request volumes or CPU contention; these may be secondary effects or separate incidents.
4. Correlate trace spans with logs (search for errors/timeouts) for the high-severity trace spikes (severity >> 5), as those are most likely to represent actual functional slowdowns rather than noise.

Bottom line
- Primary suspects: ts-consign-price-service (highest trace severity) and ts-auth-service (second-highest trace severity). Both show very large trace-duration anomalies at ~16:15 UTC and should be investigated first as potential root causes or heavily-impacted services in this incident. Secondary suspects with linked anomalies: ts-consign-service, ts-preserve-service, ts-order-service, ts-price-service. Admin/service CPU spikes exist but appear more local and less trace-linked.

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'ts-consign-price-service', 'metric_faults': [{'kpi_name': 'workload', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.533, 'global_p95': 0.37007999999999996, 'severity_ratio': 1.4402291396454823}, {'kpi_name': 'latency-90', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:16:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.0371949815075081, 'global_p95': 0.029837268222222192, 'severity_ratio': 1.2465947361697822}, {'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:03:00Z', '2024-01-24T16:19:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 0.9996000543094729, 'global_p95': 0.8656819872248664, 'severity_ratio': 1.154696608062633}, {'kpi_name': 'latency-50', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:00:00Z', '2024-01-24T16:05:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:00:00Z', 'max_value_in_window': 0.01534375, 'global_p95': 0.014384745763902739, 'severity_ratio': 1.0666681394191755}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 1251295.271641555, 'global_p95': 1189631.26913614, 'severity_ratio': 1.051834550843803}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:07:00Z', '2024-01-24T16:19:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:07:00Z', 'max_value_in_window': 11.55, 'global_p95': 11.4, 'severity_ratio': 1.013157894736842}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:20:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:20:00Z', 'max_value_in_window': 239242786.13333333, 'global_p95': 239181824.0, 'severity_ratio': 1.0002548777842473}], 'trace_faults': [{'trace_name': 'trace.from_ts-consign-price-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.1651844, 'global_p95': 0.006428089999999896, 'severity_ratio': 25.697275551525053}, {'trace_name': 'trace.to_ts-consign-price-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.1651844, 'global_p95': 0.006428089999999896, 'severity_ratio': 25.697275551525053}, {'trace_name': 'trace.from_ts-consign-price-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.0202348888888888, 'global_p95': 0.00303266666666666, 'severity_ratio': 6.6723089323660725}, {'trace_name': 'trace.to_ts-consign-price-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.0202348888888888, 'global_p95': 0.00303266666666666, 'severity_ratio': 6.6723089323660725}, {'trace_name': 'trace.from_ts-consign-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:12:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 0.1061623999999999, 'global_p95': 0.017894749999999918, 'severity_ratio': 5.93260034367624}, {'trace_name': 'trace.from_ts-consign-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:02:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:02:00Z', 'max_value_in_window': 0.0259088888888888, 'global_p95': 0.009935939682539599, 'severity_ratio': 2.607593213797224}, {'trace_name': 'trace.from_ts-consign-price-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:14:00Z', '2024-01-24T16:17:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:14:00Z', 'max_value_in_window': 26.0, 'global_p95': 21.599999999999994, 'severity_ratio': 1.203703703703704}, {'trace_name': 'trace.from_ts-consign-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:14:00Z', '2024-01-24T16:17:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:14:00Z', 'max_value_in_window': 13.0, 'global_p95': 10.799999999999997, 'severity_ratio': 1.203703703703704}, {'trace_name': 'trace.to_ts-consign-price-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:14:00Z', '2024-01-24T16:17:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:14:00Z', 'max_value_in_window': 26.0, 'global_p95': 21.599999999999994, 'severity_ratio': 1.203703703703704}], 'linking_traces': [], 'log_faults': []}, {'cmdb_id': 'ts-auth-service', 'metric_faults': [{'kpi_name': 'latency-90', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:20:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 1.2358987693050194, 'global_p95': 0.8627612103174604, 'severity_ratio': 1.4324922754121732}, {'kpi_name': 'latency-50', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:20:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 0.342699293181808, 'global_p95': 0.26655040912110123, 'severity_ratio': 1.285682863184465}, {'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:20:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 20.59449409711096, 'global_p95': 16.367140643872656, 'severity_ratio': 1.258282955173413}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:21:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 1289868.7015418366, 'global_p95': 1224481.7985115282, 'severity_ratio': 1.0533996529060639}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:11:00Z', '2024-01-24T16:16:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:11:00Z', 'max_value_in_window': 14.016666666666667, 'global_p95': 13.526666666666666, 'severity_ratio': 1.0362247412518484}, {'kpi_name': 'workload', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:03:00Z', '2024-01-24T16:08:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 2.954466666666667, 'global_p95': 2.8693366666666664, 'severity_ratio': 1.0296688781727719}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:20:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:20:00Z', 'max_value_in_window': 244627933.86666667, 'global_p95': 241430869.3333333, 'severity_ratio': 1.0132421530940159}], 'trace_faults': [{'trace_name': 'trace.from_ts-auth-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.1027002499999999, 'global_p95': 0.006234599999999998, 'severity_ratio': 16.472628556763855}, {'trace_name': 'trace.to_ts-auth-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.1027002499999999, 'global_p95': 0.006234599999999998, 'severity_ratio': 16.472628556763855}, {'trace_name': 'trace.from_ts-auth-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 0.0140145625, 'global_p95': 0.004661482586719059, 'severity_ratio': 3.006460335157879}, {'trace_name': 'trace.to_ts-auth-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 0.0140145625, 'global_p95': 0.004661482586719059, 'severity_ratio': 3.006460335157879}, {'trace_name': 'trace.from_root.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:20:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:20:00Z', 'max_value_in_window': 0.588913, 'global_p95': 0.4389879849952516, 'severity_ratio': 1.3415241877437036}, {'trace_name': 'trace.from_root.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:08:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:08:00Z', 'max_value_in_window': 1.2992587999999998, 'global_p95': 1.2011123499999996, 'severity_ratio': 1.0817129638205787}, {'trace_name': 'trace.from_root.row_count', 'total_points_in_window': 25, 'anomalies_count': 1, 'anomaly_timestamps': ['2024-01-24T16:03:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 90.0, 'global_p95': 85.0, 'severity_ratio': 1.0588235294117647}, {'trace_name': 'trace.from_ts-auth-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:00:00Z', '2024-01-24T16:03:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:00:00Z', 'max_value_in_window': 358.0, 'global_p95': 339.6, 'severity_ratio': 1.0541813898704357}, {'trace_name': 'trace.to_ts-auth-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:00:00Z', '2024-01-24T16:03:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:00:00Z', 'max_value_in_window': 358.0, 'global_p95': 339.6, 'severity_ratio': 1.0541813898704357}], 'linking_traces': [], 'log_faults': [{'log_name': 'log.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:00:00Z', '2024-01-24T16:03:00Z'], 'earliest_anomaly_timestamp': '2024-01-24T16:00:00Z', 'max_value_in_window': 358.0, 'global_p95': 339.6, 'severity_ratio': 1.0541813898704357}]}, {'cmdb_id': 'ts-consign-service', 'metric_faults': [{'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:18:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 1.466444289379657, 'global_p95': 1.0276210871571851, 'severity_ratio': 1.4270282185785366}, {'kpi_name': 'workload', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.5064, 'global_p95': 0.37535666666666656, 'severity_ratio': 1.3491168399833051}, {'kpi_name': 'latency-50', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:16:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 0.0350560515873015, 'global_p95': 0.03209226333333324, 'severity_ratio': 1.092352110637515}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 12.2, 'global_p95': 11.526666666666667, 'severity_ratio': 1.0584152689415847}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:02:00Z', '2024-01-24T16:08:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:02:00Z', 'max_value_in_window': 58391.452060709074, 'global_p95': 56276.68818316888, 'severity_ratio': 1.0375779731503936}, {'kpi_name': 'latency-90', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:11:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 0.0603973214285714, 'global_p95': 0.05873801587301584, 'severity_ratio': 1.0282492612474818}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 246788778.66666663, 'global_p95': 246462518.6133333, 'severity_ratio': 1.0013237714812335}], 'trace_faults': [{'trace_name': 'trace.to_ts-consign-price-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:12:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 0.1061623999999999, 'global_p95': 0.017894749999999918, 'severity_ratio': 5.93260034367624}, {'trace_name': 'trace.to_ts-consign-price-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:02:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:02:00Z', 'max_value_in_window': 0.0259088888888888, 'global_p95': 0.009935939682539599, 'severity_ratio': 2.607593213797224}, {'trace_name': 'trace.from_root.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:11:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:11:00Z', 'max_value_in_window': 0.1208891999999999, 'global_p95': 0.06577329999999997, 'severity_ratio': 1.8379676859759198}, {'trace_name': 'trace.from_ts-consign-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:19:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.0073287333333333, 'global_p95': 0.004958373333333279, 'severity_ratio': 1.4780519417658575}, {'trace_name': 'trace.to_ts-consign-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:15:00Z', '2024-01-24T16:19:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:15:00Z', 'max_value_in_window': 0.0073287333333333, 'global_p95': 0.004958373333333279, 'severity_ratio': 1.4780519417658575}, {'trace_name': 'trace.from_root.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:11:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:11:00Z', 'max_value_in_window': 0.0409988888888888, 'global_p95': 0.03337089999999995, 'severity_ratio': 1.2285820546910289}, {'trace_name': 'trace.from_ts-consign-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:00:00Z', '2024-01-24T16:19:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:00:00Z', 'max_value_in_window': 0.0171172999999999, 'global_p95': 0.014097949999999918, 'severity_ratio': 1.2141694359818271}, {'trace_name': 'trace.to_ts-consign-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:00:00Z', '2024-01-24T16:19:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:00:00Z', 'max_value_in_window': 0.0171172999999999, 'global_p95': 0.014097949999999918, 'severity_ratio': 1.2141694359818271}, {'trace_name': 'trace.from_root.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:14:00Z', '2024-01-24T16:17:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:14:00Z', 'max_value_in_window': 13.0, 'global_p95': 10.799999999999997, 'severity_ratio': 1.203703703703704}, {'trace_name': 'trace.from_ts-consign-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:14:00Z', '2024-01-24T16:17:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:14:00Z', 'max_value_in_window': 65.0, 'global_p95': 53.999999999999986, 'severity_ratio': 1.203703703703704}], 'linking_traces': [], 'log_faults': [{'log_name': 'log.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:14:00Z', '2024-01-24T16:17:00Z'], 'earliest_anomaly_timestamp': '2024-01-24T16:14:00Z', 'max_value_in_window': 91.0, 'global_p95': 75.59999999999998, 'severity_ratio': 1.203703703703704}]}, {'cmdb_id': 'ts-preserve-service', 'metric_faults': [{'kpi_name': 'workload', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:09:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:09:00Z', 'max_value_in_window': 0.33235, 'global_p95': 0.23283666666666655, 'severity_ratio': 1.4273954560421476}, {'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:09:00Z', '2024-01-24T16:10:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:09:00Z', 'max_value_in_window': 1.1788351298418087, 'global_p95': 1.0031203056197882, 'severity_ratio': 1.175168245760366}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:09:00Z', '2024-01-24T16:20:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:09:00Z', 'max_value_in_window': 9.916666666666666, 'global_p95': 9.079999999999998, 'severity_ratio': 1.0921439060205582}, {'kpi_name': 'latency-90', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:17:00Z', '2024-01-24T16:21:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:17:00Z', 'max_value_in_window': 1.9825, 'global_p95': 1.863916666666666, 'severity_ratio': 1.0636205123619622}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:04:00Z', '2024-01-24T16:21:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:04:00Z', 'max_value_in_window': 59159.704429468, 'global_p95': 57595.81515463265, 'severity_ratio': 1.0271528282156723}, {'kpi_name': 'latency-50', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:17:00Z', '2024-01-24T16:21:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:17:00Z', 'max_value_in_window': 0.8820138888888889, 'global_p95': 0.8589642857142856, 'severity_ratio': 1.0268341810689323}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:22:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:22:00Z', 'max_value_in_window': 226932326.4, 'global_p95': 226787614.71999997, 'severity_ratio': 1.0006380933993186}], 'trace_faults': [{'trace_name': 'trace.to_ts-seat-service.duration_p95', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:17:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:17:00Z', 'max_value_in_window': 1.0623941, 'global_p95': 0.09910366499999991, 'severity_ratio': 10.720028366256695}, {'trace_name': 'trace.to_ts-seat-service.duration_mean', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:17:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:17:00Z', 'max_value_in_window': 0.426696, 'global_p95': 0.08041398749999999, 'severity_ratio': 5.3062410317608}, {'trace_name': 'trace.to_ts-assurance-service.duration_p95', 'total_points_in_window': 21, 'anomalies_count': 1, 'anomaly_timestamps': ['2024-01-24T16:12:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 0.0750913999999999, 'global_p95': 0.01881565, 'severity_ratio': 3.990901191295539}, {'trace_name': 'trace.to_ts-assurance-service.duration_mean', 'total_points_in_window': 21, 'anomalies_count': 1, 'anomaly_timestamps': ['2024-01-24T16:12:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 0.043946, 'global_p95': 0.0151314999999999, 'severity_ratio': 2.904272544030684}, {'trace_name': 'trace.to_ts-ticketinfo-service.duration_p95', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:11:00Z', '2024-01-24T16:16:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:11:00Z', 'max_value_in_window': 0.7751329999999998, 'global_p95': 0.5074981399999989, 'severity_ratio': 1.5273612628412816}, {'trace_name': 'trace.to_ts-ticketinfo-service.duration_mean', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:11:00Z', '2024-01-24T16:16:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:11:00Z', 'max_value_in_window': 0.2689254, 'global_p95': 0.19858151749999975, 'severity_ratio': 1.3542317703358286}, {'trace_name': 'trace.to_ts-order-service.duration_p95', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:03:00Z', '2024-01-24T16:05:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 0.0869077999999999, 'global_p95': 0.0665107099999999, 'severity_ratio': 1.3066737672774809}, {'trace_name': 'trace.to_ts-order-service.row_count', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:08:00Z', '2024-01-24T16:09:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:08:00Z', 'max_value_in_window': 10.0, 'global_p95': 7.849999999999998, 'severity_ratio': 1.2738853503184717}, {'trace_name': 'trace.to_ts-seat-service.row_count', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:08:00Z', '2024-01-24T16:09:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:08:00Z', 'max_value_in_window': 10.0, 'global_p95': 7.849999999999998, 'severity_ratio': 1.2738853503184717}, {'trace_name': 'trace.to_ts-station-service.row_count', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:08:00Z', '2024-01-24T16:09:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:08:00Z', 'max_value_in_window': 20.0, 'global_p95': 15.699999999999996, 'severity_ratio': 1.2738853503184717}], 'linking_traces': [], 'log_faults': [{'log_name': 'log.row_count', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:08:00Z', '2024-01-24T16:09:00Z'], 'earliest_anomaly_timestamp': '2024-01-24T16:08:00Z', 'max_value_in_window': 265.0, 'global_p95': 228.6999999999999, 'severity_ratio': 1.1587232181897686}]}, {'cmdb_id': 'ts-order-service', 'metric_faults': [{'kpi_name': 'latency-90', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:02:00Z', '2024-01-24T16:17:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:02:00Z', 'max_value_in_window': 0.0805595022522011, 'global_p95': 0.04932259959665208, 'severity_ratio': 1.6333182539240962}, {'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:04:00Z', '2024-01-24T16:06:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:04:00Z', 'max_value_in_window': 5.343594415794218, 'global_p95': 4.676718626201181, 'severity_ratio': 1.1425948069351202}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 1285689.007204057, 'global_p95': 1240567.063281765, 'severity_ratio': 1.036372031192677}, {'kpi_name': 'latency-50', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:20:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:20:00Z', 'max_value_in_window': 0.0085622205242857, 'global_p95': 0.00847466676607502, 'severity_ratio': 1.010331233147853}, {'kpi_name': 'workload', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:09:00Z', '2024-01-24T16:10:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:09:00Z', 'max_value_in_window': 7.0353666666666665, 'global_p95': 6.992326666666666, 'severity_ratio': 1.0061553188304513}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 1, 'anomaly_timestamps': ['2024-01-24T16:09:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:09:00Z', 'max_value_in_window': 13.85, 'global_p95': 13.766666666666667, 'severity_ratio': 1.0060532687651331}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:22:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:22:00Z', 'max_value_in_window': 263308083.2, 'global_p95': 263253401.60000002, 'severity_ratio': 1.00020771469492}], 'trace_faults': [{'trace_name': 'trace.from_ts-travel-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 0.1355786, 'global_p95': 0.017738839999999888, 'severity_ratio': 7.643036410498142}, {'trace_name': 'trace.from_ts-order-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:17:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:17:00Z', 'max_value_in_window': 0.0803632999999998, 'global_p95': 0.012581759999999994, 'severity_ratio': 6.387286039473002}, {'trace_name': 'trace.to_ts-order-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:17:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:17:00Z', 'max_value_in_window': 0.0803632999999998, 'global_p95': 0.012581759999999994, 'severity_ratio': 6.387286039473002}, {'trace_name': 'trace.from_ts-travel-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 0.0371533999999999, 'global_p95': 0.009334866585067233, 'severity_ratio': 3.980067595120569}, {'trace_name': 'trace.from_ts-inside-payment-service.duration_mean', 'total_points_in_window': 23, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:01:00Z', '2024-01-24T16:21:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:01:00Z', 'max_value_in_window': 0.028836, 'global_p95': 0.012844848571428508, 'severity_ratio': 2.244946667891553}, {'trace_name': 'trace.from_ts-inside-payment-service.duration_p95', 'total_points_in_window': 23, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:01:00Z', '2024-01-24T16:21:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:01:00Z', 'max_value_in_window': 0.0710244999999999, 'global_p95': 0.04086954999999985, 'severity_ratio': 1.7378341577042113}, {'trace_name': 'trace.from_ts-seat-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:14:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 0.0141018375, 'global_p95': 0.010167107008546958, 'severity_ratio': 1.387005909168195}, {'trace_name': 'trace.from_ts-preserve-service.duration_p95', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:03:00Z', '2024-01-24T16:05:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 0.0869077999999999, 'global_p95': 0.0665107099999999, 'severity_ratio': 1.3066737672774809}, {'trace_name': 'trace.from_ts-preserve-service.row_count', 'total_points_in_window': 24, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:08:00Z', '2024-01-24T16:09:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:08:00Z', 'max_value_in_window': 10.0, 'global_p95': 7.849999999999998, 'severity_ratio': 1.2738853503184717}, {'trace_name': 'trace.from_ts-seat-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:01:00Z', '2024-01-24T16:05:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:01:00Z', 'max_value_in_window': 0.0511236499999999, 'global_p95': 0.04246783999999985, 'severity_ratio': 1.2038203497046256}], 'linking_traces': [], 'log_faults': [{'log_name': 'log.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:06:00Z', '2024-01-24T16:09:00Z'], 'earliest_anomaly_timestamp': '2024-01-24T16:06:00Z', 'max_value_in_window': 470.0, 'global_p95': 448.2, 'severity_ratio': 1.0486390004462294}]}, {'cmdb_id': 'ts-admin-user-service', 'metric_faults': [{'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:13:00Z', '2024-01-24T16:14:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:13:00Z', 'max_value_in_window': 0.3066789008039297, 'global_p95': 0.11098184321038641, 'severity_ratio': 2.7633249902199197}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 1121086.7019887627, 'global_p95': 1092664.2821149363, 'severity_ratio': 1.02601203346632}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 3.3833333333333333, 'global_p95': 3.31, 'severity_ratio': 1.0221550855991943}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:22:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:22:00Z', 'max_value_in_window': 190300160.0, 'global_p95': 190290315.94666663, 'severity_ratio': 1.000051731761989}], 'trace_faults': [], 'linking_traces': [], 'log_faults': []}, {'cmdb_id': 'ts-admin-route-service', 'metric_faults': [{'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:05:00Z', '2024-01-24T16:14:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 0.3099755717343141, 'global_p95': 0.11720491268202671, 'severity_ratio': 2.6447319027937697}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 1133688.1160160375, 'global_p95': 1074026.8291402278, 'severity_ratio': 1.0555491587892356}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:01:00Z', '2024-01-24T16:16:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:01:00Z', 'max_value_in_window': 3.3, 'global_p95': 3.273095238095238, 'severity_ratio': 1.008219975267331}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 1, 'anomaly_timestamps': ['2024-01-24T16:18:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 196196420.26666668, 'global_p95': 196194304.0, 'severity_ratio': 1.000010786585663}], 'trace_faults': [], 'linking_traces': [], 'log_faults': []}, {'cmdb_id': 'ts-price-service', 'metric_faults': [{'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:12:00Z', '2024-01-24T16:13:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 8.093841513845785, 'global_p95': 4.480958028768699, 'severity_ratio': 1.8062747880880852}, {'kpi_name': 'workload', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T15:59:00Z', '2024-01-24T16:06:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T15:59:00Z', 'max_value_in_window': 5.654366666666666, 'global_p95': 5.329478095238096, 'severity_ratio': 1.060960673000769}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 1300995.3430349722, 'global_p95': 1288966.6067817218, 'severity_ratio': 1.0093320774874717}, {'kpi_name': 'latency-50', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:11:00Z', '2024-01-24T16:12:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:11:00Z', 'max_value_in_window': 0.0069148511519269, 'global_p95': 0.006858757723609579, 'severity_ratio': 1.008178365613387}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 1, 'anomaly_timestamps': ['2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:23:00Z', 'max_value_in_window': 18.0, 'global_p95': 17.866666666666667, 'severity_ratio': 1.007462686567164}, {'kpi_name': 'latency-90', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:12:00Z', '2024-01-24T16:20:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 0.0097634904879035, 'global_p95': 0.009700601761528439, 'severity_ratio': 1.006482971667229}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:12:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 256812578.13333333, 'global_p95': 256603968.8533333, 'severity_ratio': 1.0008129620166524}], 'trace_faults': [{'trace_name': 'trace.from_ts-admin-basic-info-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:12:00Z', '2024-01-24T16:18:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 0.0444321999999999, 'global_p95': 0.0084151299999999, 'severity_ratio': 5.280037266209843}, {'trace_name': 'trace.from_ts-admin-basic-info-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:06:00Z', '2024-01-24T16:12:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:06:00Z', 'max_value_in_window': 0.0117144444444444, 'global_p95': 0.00444299999999998, 'severity_ratio': 2.6366068972416055}, {'trace_name': 'trace.from_ts-basic-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:19:00Z', '2024-01-24T16:20:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:19:00Z', 'max_value_in_window': 0.0095381999999999, 'global_p95': 0.006528499999999899, 'severity_ratio': 1.4610094202343642}, {'trace_name': 'trace.from_ts-basic-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:03:00Z', '2024-01-24T16:06:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 172.0, 'global_p95': 146.6, 'severity_ratio': 1.1732605729877217}, {'trace_name': 'trace.from_ts-price-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:03:00Z', '2024-01-24T16:06:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 360.0, 'global_p95': 311.6, 'severity_ratio': 1.1553273427471116}, {'trace_name': 'trace.to_ts-price-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:03:00Z', '2024-01-24T16:06:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 360.0, 'global_p95': 311.6, 'severity_ratio': 1.1553273427471116}, {'trace_name': 'trace.from_ts-price-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:19:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:19:00Z', 'max_value_in_window': 0.0026500189393939, 'global_p95': 0.002308218794366099, 'severity_ratio': 1.148079612670197}, {'trace_name': 'trace.to_ts-price-service.duration_mean', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:19:00Z', '2024-01-24T16:22:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:19:00Z', 'max_value_in_window': 0.0026500189393939, 'global_p95': 0.002308218794366099, 'severity_ratio': 1.148079612670197}, {'trace_name': 'trace.from_ts-admin-basic-info-service.row_count', 'total_points_in_window': 25, 'anomalies_count': 1, 'anomaly_timestamps': ['2024-01-24T16:05:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:05:00Z', 'max_value_in_window': 11.0, 'global_p95': 10.0, 'severity_ratio': 1.1}, {'trace_name': 'trace.from_ts-price-service.duration_p95', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:12:00Z', '2024-01-24T16:19:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:12:00Z', 'max_value_in_window': 0.0027566999999999, 'global_p95': 0.0025287299999999196, 'severity_ratio': 1.0901519735202998}], 'linking_traces': [], 'log_faults': [{'log_name': 'log.row_count', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:03:00Z', '2024-01-24T16:06:00Z'], 'earliest_anomaly_timestamp': '2024-01-24T16:03:00Z', 'max_value_in_window': 344.0, 'global_p95': 293.2, 'severity_ratio': 1.1732605729877217}]}, {'cmdb_id': 'ts-route-plan-service', 'metric_faults': [{'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:13:00Z', '2024-01-24T16:14:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:13:00Z', 'max_value_in_window': 0.2671911948108849, 'global_p95': 0.10675699813478533, 'severity_ratio': 2.502797938113101}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:13:00Z', '2024-01-24T16:14:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:13:00Z', 'max_value_in_window': 54307.60150924089, 'global_p95': 52869.59557931463, 'severity_ratio': 1.0271991096994297}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:11:00Z', '2024-01-24T16:12:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:11:00Z', 'max_value_in_window': 3.25, 'global_p95': 3.193333333333333, 'severity_ratio': 1.0177453027139876}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:22:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:22:00Z', 'max_value_in_window': 189779968.0, 'global_p95': 189775175.68, 'severity_ratio': 1.0000252526179085}], 'trace_faults': [], 'linking_traces': [], 'log_faults': []}, {'cmdb_id': 'ts-travel-plan-service', 'metric_faults': [{'kpi_name': 'cpu', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:14:00Z', '2024-01-24T16:15:00Z'], 'longest_consecutive_anomaly_run_length': 2, 'earliest_anomaly_timestamp': '2024-01-24T16:14:00Z', 'max_value_in_window': 0.2887174920005124, 'global_p95': 0.11698612977238922, 'severity_ratio': 2.467963446284166}, {'kpi_name': 'socket', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:14:00Z', '2024-01-24T16:19:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:14:00Z', 'max_value_in_window': 3.433333333333333, 'global_p95': 3.3466666666666667, 'severity_ratio': 1.0258964143426295}, {'kpi_name': 'diskio', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:18:00Z', '2024-01-24T16:23:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:18:00Z', 'max_value_in_window': 1145340.1086490063, 'global_p95': 1119687.1388407906, 'severity_ratio': 1.0229108372493894}, {'kpi_name': 'mem', 'total_points_in_window': 25, 'anomalies_count': 2, 'anomaly_timestamps': ['2024-01-24T16:19:00Z', '2024-01-24T16:21:00Z'], 'longest_consecutive_anomaly_run_length': 1, 'earliest_anomaly_timestamp': '2024-01-24T16:19:00Z', 'max_value_in_window': 188676846.9333333, 'global_p95': 188675304.10666665, 'severity_ratio': 1.0000081771521396}], 'trace_faults': [], 'linking_traces': [], 'log_faults': []}]```
```

In [ ]:
# In[4]:

import re
import numpy as np
import pandas as pd

# Reuse existing variables: trace_window, trace_p95
candidates = [
    "ts-consign-price-service",
    "ts-auth-service",
    "ts-consign-service",
    "ts-preserve-service",
    "ts-order-service",
    "ts-price-service",
    "ts-preserve-other-service",
    "ts-seat-service",
    "ts-travel-service",
]

# Ensure value numeric and timestamp typed
trace_window['value'] = pd.to_numeric(trace_window['value'], errors='coerce')
# Merge global p95 from full-series thresholds
trace_with_p95 = trace_window.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')

# Keep only rows with from_ts- or to_ts- in trace_name
pattern = r'(from_ts-|to_ts-)'
trace_links = trace_with_p95[trace_with_p95['trace_name'].str.contains(pattern, regex=True)].copy()

# Parse source and target
def parse_pair(row):
    tn = row['trace_name']
    cmdb = row['cmdb_id']
    m_from = re.search(r'from_ts-([^.\s]+)', tn)
    m_to = re.search(r'to_ts-([^.\s]+)', tn)
    source = None
    target = None
    if m_from:
        source = m_from.group(1)
        target = cmdb
    elif m_to:
        source = cmdb
        target = m_to.group(1)
    return pd.Series({'source_service': source, 'target_service': target})

trace_links[['source_service', 'target_service']] = trace_links.apply(parse_pair, axis=1)

# Filter to rows where source or target is in the candidate list
trace_links_candidates = trace_links[
    trace_links['source_service'].isin(candidates) | trace_links['target_service'].isin(candidates)
].copy()

# Determine anomalies: value > p95 (p95 may be NaN -> not anomaly)
trace_links_candidates['is_anomaly'] = False
# Only mark anomaly when both value and p95 present and value > p95
mask_valid = trace_links_candidates['p95'].notna() & trace_links_candidates['value'].notna()
trace_links_candidates.loc[mask_valid, 'is_anomaly'] = trace_links_candidates.loc[mask_valid, 'value'] > trace_links_candidates.loc[mask_valid, 'p95']

# Compute severity ratio (value / p95) where p95>0, else None
def compute_sev(v, p):
    if pd.isna(v) or pd.isna(p):
        return None
    if p == 0:
        return None
    return float(v / p)

trace_links_candidates['severity_ratio'] = trace_links_candidates.apply(lambda r: compute_sev(r['value'], r['p95']), axis=1)

# Format timestamp for output convenience
trace_links_candidates['ts_iso'] = trace_links_candidates['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Group by (source_service, target_service) and compute required aggregates
groups = []
grouped = trace_links_candidates.groupby(['source_service', 'target_service'], dropna=False)

for (src, tgt), g in grouped:
    total_points = int(g.shape[0])
    anomalies = g[g['is_anomaly']]
    anomalies_count = int(anomalies.shape[0])
    if anomalies_count > 0:
        earliest_anom_ts = anomalies['timestamp'].min().strftime('%Y-%m-%dT%H:%M:%SZ')
        # compute max severity among anomalies (handle None)
        sev_vals = anomalies['severity_ratio'].dropna().values
        max_sev = float(np.max(sev_vals)) if len(sev_vals) > 0 else None
        # sample up to 5 anomalous rows ordered by timestamp asc
        sample_rows = anomalies.sort_values('timestamp', ascending=True).head(5)
        sample_anomalous_rows = []
        for _, r in sample_rows.iterrows():
            sample_anomalous_rows.append({
                'timestamp': r['ts_iso'],
                'cmdb_id': r['cmdb_id'],
                'trace_name': r['trace_name'],
                'value': float(r['value']) if not pd.isna(r['value']) else None,
                'global_p95': float(r['p95']) if not pd.isna(r['p95']) else None,
                'severity_ratio': r['severity_ratio']
            })
    else:
        earliest_anom_ts = None
        max_sev = None
        sample_anomalous_rows = []

    groups.append({
        'source_service': src,
        'target_service': tgt,
        'total_points_in_window': total_points,
        'anomalies_count': anomalies_count,
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'max_severity_ratio': max_sev,
        'sample_anomalous_rows': sample_anomalous_rows
    })

# Build DataFrame
pairs_df = pd.DataFrame(groups)

# Filter to pairs that have source/target in candidates and total_points>0 (already so), keep only pairs where either source or target not null
pairs_df = pairs_df.dropna(subset=['source_service', 'target_service'])

# Count unique pairs with anomalies_count>0
unique_pairs_with_anomalies_count = int((pairs_df['anomalies_count'] > 0).sum())

# Sort by max_severity_ratio desc (treat None as -inf) then anomalies_count desc
pairs_df['sort_sev'] = pairs_df['max_severity_ratio'].apply(lambda x: x if x is not None else -np.inf)
pairs_df = pairs_df.sort_values(['sort_sev', 'anomalies_count'], ascending=[False, False]).drop(columns=['sort_sev']).reset_index(drop=True)

# Limit to top 20 rows
result_top20 = pairs_df.head(20).copy()

# Keep output compact: selected columns and sample_anomalous_rows as-is (list)
result_top20 = result_top20[[
    'source_service',
    'target_service',
    'total_points_in_window',
    'anomalies_count',
    'earliest_anomaly_timestamp',
    'max_severity_ratio',
    'sample_anomalous_rows'
]]

# Final outputs (JSON-serializable compatible)
result_top20, unique_pairs_with_anomalies_count

```
Out[4]:
```
Summary of inter-service trace links among the candidate services (window 2024-01-24 15:56:04 → 2024-01-24 16:26:04 UTC)

Top linked (source → target) pairs by peak severity (max value / global P95) — showing total rows, anomaly count, earliest anomaly time, and peak severity:
1. consign-price-service → ts-consign-price-service
   - total points: 100, anomalies: 6, earliest anomaly: 2024-01-24T16:14:00Z
   - max severity ≈ 25.70
2. ts-consign-price-service → consign-price-service
   - total points: 100, anomalies: 6, earliest anomaly: 2024-01-24T16:14:00Z
   - max severity ≈ 25.70
3. auth-service → ts-auth-service
   - total points: 100, anomalies: 6, earliest anomaly: 2024-01-24T16:00:00Z
   - max severity ≈ 16.47
4. ts-auth-service → auth-service
   - same counts and earliest anomaly 2024-01-24T16:00:00Z
   - max severity ≈ 16.47
5. preserve-service → ts-seat-service
   - total points: 96, anomalies: 6, earliest anomaly: 2024-01-24T16:08:00Z
   - max severity ≈ 10.72
6. ts-preserve-service → seat-service
   - same totals and earliest anomaly 2024-01-24T16:08:00Z
   - max severity ≈ 10.72
7. travel-service → ts-order-service
   - total points: 100, anomalies: 6, earliest anomaly: 2024-01-24T15:59:00Z
   - max severity ≈ 7.64
8. ts-travel-service → order-service
   - same totals and earliest anomaly 2024-01-24T15:59:00Z
   - max severity ≈ 7.64
9. admin-basic-info-service → ts-price-service
   - total points: 100, anomalies: 5, earliest anomaly: 2024-01-24T16:05:00Z
   - max severity ≈ 5.28
10. consign-service → ts-consign-price-service
    - total points: 100, anomalies: 6, earliest anomaly: 2024-01-24T16:02:00Z
    - max severity ≈ 5.93

(Other pairs with lower peak severities appear next; the full top-20 includes links involving order-service, price-service, preserve-other, seat-service and others. A total of 66 unique (source,target) pairs had anomalies_count > 0.)

Key observations and implications
- Two strongest linkages are the consign-price ↔ ts-consign-price pair (peak severity ≈ 25.7 at ~16:14) and the auth ↔ ts-auth pair (peak severity ≈ 16.47 at ~16:00). These are extreme relative deviations and are the highest-priority traces to inspect.
- Many candidate-to-candidate links are bi-directional (both "from_ts-..." and "to_ts-..." forms exist), indicating the slow spans appear both in outgoing and incoming traces around the same times — consistent with large service-to-service latency rather than a single isolated sampling artifact.
- preserve-service → ts-seat-service shows another very large spike (severity ≈ 10.7, earliest ~16:08), suggesting a separate high-impact chain (preserve -> seat).
- travel → order traces show elevated severity earlier (~15:59), so some latency propagation may start before the main 16:14 consign-price spike.
- The presence of many anomalous (source,target) pairs (66 unique with anomalies) indicates the issue is affecting multiple inter-service flows and is not restricted to a single trivial link.

Actionable next steps (recommended)
1. Pull full span-level traces (trace/span details) for:
   - consign-price ↔ ts-consign-price around 16:12–16:16 and 16:20–16:23 UTC,
   - auth ↔ ts-auth around 16:00–16:05 and 16:15–16:23 UTC,
   - preserve → seat around 16:07–16:18 UTC,
   - travel → order around ~15:59–16:05 UTC.
   Inspect span durations, error flags, and downstream calls to see whether latency originates in the source, target, or in a downstream dependency (DB, cache, external API).
2. Correlate these slow spans with service logs and resource metrics at the matching timestamps (CPU, thread pools, DB latency, connection errors) for the implicated services (ts-consign-price-service, ts-auth-service, ts-preserve-service, ts-order-service, ts-price-service).
3. Prioritize ts-consign-price-service and ts-auth-service for immediate investigative triage given their very high severity ratios and early anomaly times.

If you want, I can extract the up-to-5 sample anomalous rows for any specific (source,target) pair in the top list to show exact timestamps and values for quick forensic checks.

The original code execution output of IPython Kernel is also provided below for reference:

(               source_service            target_service  total_points_in_window  anomalies_count earliest_anomaly_timestamp  max_severity_ratio                              sample_anomalous_rows
0       consign-price-service  ts-consign-price-service                     100                6       2024-01-24T16:14:00Z           25.697276  [{'timestamp': '2024-01-24T16:14:00Z', 'cmdb_i...
1    ts-consign-price-service     consign-price-service                     100                6       2024-01-24T16:14:00Z           25.697276  [{'timestamp': '2024-01-24T16:14:00Z', 'cmdb_i...
2                auth-service           ts-auth-service                     100                6       2024-01-24T16:00:00Z           16.472629  [{'timestamp': '2024-01-24T16:00:00Z', 'cmdb_i...
3             ts-auth-service              auth-service                     100                6       2024-01-24T16:00:00Z           16.472629  [{'timestamp': '2024-01-24T16:00:00Z', 'cmdb_i...
4            preserve-service           ts-seat-service                      96                6       2024-01-24T16:08:00Z           10.720028  [{'timestamp': '2024-01-24T16:08:00Z', 'cmdb_i...
5         ts-preserve-service              seat-service                      96                6       2024-01-24T16:08:00Z           10.720028  [{'timestamp': '2024-01-24T16:08:00Z', 'cmdb_i...
6              travel-service          ts-order-service                     100                6       2024-01-24T15:59:00Z            7.643036  [{'timestamp': '2024-01-24T15:59:00Z', 'cmdb_i...
7           ts-travel-service             order-service                     100                6       2024-01-24T15:59:00Z            7.643036  [{'timestamp': '2024-01-24T15:59:00Z', 'cmdb_i...
8   ts-preserve-other-service        ticketinfo-service                      96                5       2024-01-24T16:06:00Z            6.639893  [{'timestamp': '2024-01-24T16:06:00Z', 'cmdb_i...
9               order-service          ts-order-service                     100                6       2024-01-24T16:06:00Z            6.387286  [{'timestamp': '2024-01-24T16:06:00Z', 'cmdb_i...
10           ts-order-service             order-service                     100                6       2024-01-24T16:06:00Z            6.387286  [{'timestamp': '2024-01-24T16:06:00Z', 'cmdb_i...
11            consign-service  ts-consign-price-service                     100                6       2024-01-24T16:02:00Z            5.932600  [{'timestamp': '2024-01-24T16:02:00Z', 'cmdb_i...
12         ts-consign-service     consign-price-service                     100                6       2024-01-24T16:02:00Z            5.932600  [{'timestamp': '2024-01-24T16:02:00Z', 'cmdb_i...
13   admin-basic-info-service          ts-price-service                     100                5       2024-01-24T16:05:00Z            5.280037  [{'timestamp': '2024-01-24T16:05:00Z', 'cmdb_i...
14        ts-preserve-service         assurance-service                      84                2       2024-01-24T16:12:00Z            3.990901  [{'timestamp': '2024-01-24T16:12:00Z', 'cmdb_i...
15  ts-preserve-other-service              user-service                      96                5       2024-01-24T16:03:00Z            2.832561  [{'timestamp': '2024-01-24T16:03:00Z', 'cmdb_i...
16            ts-seat-service       order-other-service                     100                6       2024-01-24T16:06:00Z            2.398377  [{'timestamp': '2024-01-24T16:06:00Z', 'cmdb_i...
17     inside-payment-service          ts-order-service                      92                6       2024-01-24T16:01:00Z            2.244947  [{'timestamp': '2024-01-24T16:01:00Z', 'cmdb_i...
18  ts-preserve-other-service       order-other-service                      96                5       2024-01-24T16:03:00Z            1.863090  [{'timestamp': '2024-01-24T16:03:00Z', 'cmdb_i...
19  ts-preserve-other-service              food-service                      72                3       2024-01-24T16:06:00Z            1.739130  [{'timestamp': '2024-01-24T16:06:00Z', 'cmdb_i..., 66)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd

# Reuse previously computed dataframes and p95s:
# metric, trace, log, metric_p95, trace_p95, log_p95, metric_window, trace_window, log_window
# candidates list provided
candidates = [
    "ts-consign-price-service",
    "ts-auth-service",
    "ts-consign-service",
    "ts-preserve-service",
    "ts-order-service",
    "ts-price-service",
    "ts-preserve-other-service",
    "ts-seat-service",
    "ts-travel-service",
]

# Helper (reuse if available)
try:
    longest_consecutive_run
except NameError:
    def longest_consecutive_run(tstamps):
        if len(tstamps) == 0:
            return 0
        secs = np.array([int(pd.to_datetime(ts).value // 10**9) for ts in tstamps], dtype=np.int64)
        if len(secs) == 1:
            return 1
        diffs = np.diff(secs)
        runs = []
        run_len = 1
        for d in diffs:
            if d == 60:
                run_len += 1
            else:
                runs.append(run_len)
                run_len = 1
        runs.append(run_len)
        return int(max(runs)) if runs else int(run_len)

# Prepare anomaly rows for each telemetry type limited to candidates and within window
# Metric
metric_window['value'] = pd.to_numeric(metric_window['value'], errors='coerce')
metric_merge = metric_window.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')
mask_valid = metric_merge['p95'].notna() & metric_merge['value'].notna()
metric_merge['is_anomaly'] = False
metric_merge.loc[mask_valid, 'is_anomaly'] = metric_merge.loc[mask_valid, 'value'] > metric_merge.loc[mask_valid, 'p95']
def sev_val(v,p):
    if pd.isna(v) or pd.isna(p): return None
    if p == 0: return None
    return float(v / p)
metric_merge['severity_ratio'] = metric_merge.apply(lambda r: sev_val(r['value'], r['p95']), axis=1)
metric_anom = metric_merge[(metric_merge['cmdb_id'].isin(candidates)) & (metric_merge['is_anomaly'])].copy()

# Trace
trace_window['value'] = pd.to_numeric(trace_window['value'], errors='coerce')
trace_merge = trace_window.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')
mask_valid = trace_merge['p95'].notna() & trace_merge['value'].notna()
trace_merge['is_anomaly'] = False
trace_merge.loc[mask_valid, 'is_anomaly'] = trace_merge.loc[mask_valid, 'value'] > trace_merge.loc[mask_valid, 'p95']
trace_merge['severity_ratio'] = trace_merge.apply(lambda r: sev_val(r['value'], r['p95']), axis=1)
trace_anom = trace_merge[(trace_merge['cmdb_id'].isin(candidates)) & (trace_merge['is_anomaly'])].copy()

# Log
log_window['value'] = pd.to_numeric(log_window['value'], errors='coerce')
log_merge = log_window.merge(log_p95, on=['cmdb_id', 'log_name'], how='left')
mask_valid = log_merge['p95'].notna() & log_merge['value'].notna()
log_merge['is_anomaly'] = False
log_merge.loc[mask_valid, 'is_anomaly'] = log_merge.loc[mask_valid, 'value'] > log_merge.loc[mask_valid, 'p95']
log_merge['severity_ratio'] = log_merge.apply(lambda r: sev_val(r['value'], r['p95']), axis=1)
log_anom = log_merge[(log_merge['cmdb_id'].isin(candidates)) & (log_merge['is_anomaly'])].copy()

# Compute earliest anomaly per telemetry per service
def earliest_by(df, time_col='timestamp'):
    if df.empty:
        return {}
    out = df.groupby('cmdb_id')[time_col].min().to_dict()
    return out

earliest_metric = earliest_by(metric_anom)  # dict cmdb_id->Timestamp
earliest_trace = earliest_by(trace_anom)
earliest_log = earliest_by(log_anom)

# Compute max severity per telemetry per service (exclude None)
def max_sev_by(df):
    if df.empty:
        return {}
    # convert None to NaN for numeric aggregation
    s = df[['cmdb_id','severity_ratio']].copy()
    s['severity_numeric'] = s['severity_ratio'].apply(lambda x: np.nan if x is None else x)
    out = s.groupby('cmdb_id')['severity_numeric'].max().to_dict()
    # convert nan to None
    return {k:(None if pd.isna(v) else float(v)) for k,v in out.items()}

maxsev_metric = max_sev_by(metric_anom)
maxsev_trace = max_sev_by(trace_anom)
maxsev_log = max_sev_by(log_anom)

# Compute longest consecutive run per series for each telemetry then take per-service max
series_runs = []

# Metric series runs: per (cmdb_id, kpi_name)
for (cid, kpi), g in metric_anom.groupby(['cmdb_id','kpi_name']):
    ts_list = pd.to_datetime(g['timestamp']).sort_values().unique()
    run = longest_consecutive_run(ts_list)
    series_runs.append({'cmdb_id': cid, 'series': kpi, 'run': run})
# Trace series runs
for (cid, tname), g in trace_anom.groupby(['cmdb_id','trace_name']):
    ts_list = pd.to_datetime(g['timestamp']).sort_values().unique()
    run = longest_consecutive_run(ts_list)
    series_runs.append({'cmdb_id': cid, 'series': tname, 'run': run})
# Log series runs
for (cid, lname), g in log_anom.groupby(['cmdb_id','log_name']):
    ts_list = pd.to_datetime(g['timestamp']).sort_values().unique()
    run = longest_consecutive_run(ts_list)
    series_runs.append({'cmdb_id': cid, 'series': lname, 'run': run})

series_runs_df = pd.DataFrame(series_runs)
if series_runs_df.empty:
    per_service_longest = {}
else:
    per_service_longest = series_runs_df.groupby('cmdb_id')['run'].max().to_dict()

# Total anomalous series count per service (distinct series across metric/trace/log)
anom_series_sets = {}
for cid in candidates:
    s = set()
    if not metric_anom[metric_anom['cmdb_id']==cid].empty:
        s.update(metric_anom[metric_anom['cmdb_id']==cid]['kpi_name'].unique().tolist())
    if not trace_anom[trace_anom['cmdb_id']==cid].empty:
        s.update(trace_anom[trace_anom['cmdb_id']==cid]['trace_name'].unique().tolist())
    if not log_anom[log_anom['cmdb_id']==cid].empty:
        s.update(log_anom[log_anom['cmdb_id']==cid]['log_name'].unique().tolist())
    anom_series_sets[cid] = s

# Assemble final per-service summary
rows = []
for cid in candidates:
    # earliest among types with tie-breaker trace > metric > log
    em = earliest_trace.get(cid, None)
    ep = earliest_metric.get(cid, None)
    el = earliest_log.get(cid, None)
    # Build list of (timestamp, type_priority, type_name) where priority: trace=1, metric=2, log=3
    cand_times = []
    if not pd.isna(em) and em is not None:
        cand_times.append((pd.to_datetime(em), 1, 'trace'))
    if not pd.isna(ep) and ep is not None:
        cand_times.append((pd.to_datetime(ep), 2, 'metric'))
    if not pd.isna(el) and el is not None:
        cand_times.append((pd.to_datetime(el), 3, 'log'))
    if len(cand_times) == 0:
        earliest_ts = None
        earliest_type = None
    else:
        # sort by timestamp asc then priority asc
        cand_times.sort(key=lambda x: (x[0], x[1]))
        chosen = cand_times[0]
        earliest_ts = chosen[0].strftime('%Y-%m-%dT%H:%M:%SZ')
        earliest_type = chosen[2]
    # max severity across telemetry
    sev_vals = []
    for d in (maxsev_trace, maxsev_metric, maxsev_log):
        v = d.get(cid, None)
        if v is not None:
            sev_vals.append(v)
    # remove None
    sev_vals = [v for v in sev_vals if v is not None]
    max_sev = float(max(sev_vals)) if len(sev_vals)>0 else None
    # longest run
    longest_run = int(per_service_longest.get(cid, 0)) if cid in per_service_longest else 0
    total_anomalous_series_count = len(anom_series_sets.get(cid, set()))
    rows.append({
        'cmdb_id': cid,
        'earliest_anomaly_timestamp': earliest_ts,
        'earliest_telemetry_type': earliest_type,
        'max_severity_ratio': max_sev,
        'longest_consecutive_fault_length': longest_run,
        'total_anomalous_series_count': int(total_anomalous_series_count)
    })

service_summary = pd.DataFrame(rows)

# Sort by earliest_anomaly_timestamp ascending (None last) then max_severity_ratio desc (None -> -inf)
def sort_key(df):
    # convert earliest to datetime or far future for None
    df2 = df.copy()
    df2['earliest_dt'] = pd.to_datetime(df2['earliest_anomaly_timestamp'], utc=True, errors='coerce')
    df2['earliest_sort'] = df2['earliest_dt'].fillna(pd.Timestamp.max.tz_localize('UTC'))
    df2['max_sev_sort'] = df2['max_severity_ratio'].apply(lambda x: x if x is not None else -np.inf)
    df_sorted = df2.sort_values(['earliest_sort','max_sev_sort'], ascending=[True, False]).drop(columns=['earliest_dt','earliest_sort','max_sev_sort'])
    return df_sorted

service_summary = sort_key(service_summary).reset_index(drop=True)

# Limit rows to candidates (already) and <=20, show selected columns
service_summary = service_summary[['cmdb_id','earliest_anomaly_timestamp','earliest_telemetry_type','max_severity_ratio','longest_consecutive_fault_length','total_anomalous_series_count']].head(20)

service_summary

```
Out[5]:
```
Summary (per-service within window 2024-01-24 15:56:04 → 2024-01-24 16:26:04 UTC)

Table (compact verbal form)
- ts-preserve-service — earliest anomaly 2024-01-24T15:59:00Z (trace); max severity ≈ 10.72; longest consecutive run = 2 minutes; 45 distinct anomalous series.
- ts-seat-service — earliest anomaly 2024-01-24T15:59:00Z (trace); max severity ≈ 10.72; longest run = 2 min; 41 anomalous series.
- ts-order-service — earliest anomaly 2024-01-24T15:59:00Z (trace); max severity ≈ 7.64; longest run = 2 min; 35 anomalous series.
- ts-travel-service — earliest anomaly 2024-01-24T15:59:00Z (trace); max severity ≈ 7.64; longest run = 2 min; 44 anomalous series.
- ts-preserve-other-service — earliest anomaly 2024-01-24T15:59:00Z (trace); max severity ≈ 6.64; longest run = 2 min; 46 anomalous series.
- ts-price-service — earliest anomaly 2024-01-24T15:59:00Z (metric); max severity ≈ 5.28; longest run = 2 min; 20 anomalous series.
- ts-consign-price-service — earliest anomaly 2024-01-24T16:00:00Z (metric); max severity ≈ 25.70 (largest overall); longest run = 2 min; 16 anomalous series.
- ts-auth-service — earliest anomaly 2024-01-24T16:00:00Z (trace); max severity ≈ 16.47; longest run = 1 min; 17 anomalous series.
- ts-consign-service — earliest anomaly 2024-01-24T16:00:00Z (trace); max severity ≈ 5.93; longest run = 1 min; 20 anomalous series.

Key takeaways
- Earliest anomalies (15:59Z) appear in a cluster of trace-linked services: preserve / seat / order / travel / preserve-other (trace telemetry). These may represent the initial propagation of latency across flows.
- The largest severity ratios are from ts-consign-price-service (≈25.7) and ts-auth-service (≈16.47) (both anomalies at ~16:00). These are the strongest single‑series deviations from their global P95s and are highest-priority for root-cause investigation.
- Several services show short consecutive runs (1–2 minutes) of anomalies rather than long sustained runs; many distinct series per service are affected (tens of anomalous series), indicating multi-dimensional impact.
- Interpretation: an incident appears to begin ~15:59 UTC across several trace flows and then exhibits very large service-level trace spikes around ~16:00–16:15 UTC, with ts-consign-price-service and ts-auth-service being the most extreme outliers.

Recommended next steps
- Immediately inspect span-level traces and service logs for ts-consign-price-service and ts-auth-service around 16:00–16:16 UTC.
- Correlate those with trace flows involving preserve/seat/order/travel (earliest 15:59Z) to determine whether the problem originated there or propagated from another dependency (DB, cache, or external).
- Check resource metrics (CPU, threads, DB latency) for the high-severity services at the listed timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_anomaly_timestamp earliest_telemetry_type  max_severity_ratio  longest_consecutive_fault_length  total_anomalous_series_count
0        ts-preserve-service       2024-01-24T15:59:00Z                   trace           10.720028                                 2                            45
1            ts-seat-service       2024-01-24T15:59:00Z                   trace           10.720028                                 2                            41
2           ts-order-service       2024-01-24T15:59:00Z                   trace            7.643036                                 2                            35
3          ts-travel-service       2024-01-24T15:59:00Z                   trace            7.643036                                 2                            44
4  ts-preserve-other-service       2024-01-24T15:59:00Z                   trace            6.639893                                 2                            46
5           ts-price-service       2024-01-24T15:59:00Z                  metric            5.280037                                 2                            20
6   ts-consign-price-service       2024-01-24T16:00:00Z                  metric           25.697276                                 2                            16
7            ts-auth-service       2024-01-24T16:00:00Z                   trace           16.472629                                 1                            17
8         ts-consign-service       2024-01-24T16:00:00Z                   trace            5.932600                                 1                            20```
```